## Environment setup

The original IPython history starts with this shell command for the analysis and plotting libraries:

```powershell
python -m pip install pillow matplotlib
```

Run it in a terminal if those packages are missing. The training dependencies are also listed in the project requirements file.


In [ ]:
# ============================================================
# Part 1: Import the Python libraries that we need.
# These libraries help us find files, read JSON data, and crop images.
# ============================================================

import os
import json
import math
from pathlib import Path
from PIL import Image


# ============================================================
# Part 2: Set the folders where the downloaded dataset is stored.
# Please check these paths on your own computer before running the code.
# ============================================================

desktop_path = Path.home() / "Desktop"

# The first training image folder.
train_image_folder_0 = desktop_path / "data" / "images"

# The second training image folder.
train_image_folder_1 = desktop_path / "Data 2" / "images"

# The third training image folder.
train_image_folder_2 = desktop_path / "Data3" / "images"

# The folder containing the JSON annotation files.
annotation_folder = (
    desktop_path
    / "Annotations"
    / "mtsd_v2_fully_annotated"
    / "annotations"
)

# The folder where the cropped traffic signs will be saved.
output_folder = desktop_path / "Crop_Test"

# Create the output folder if it does not already exist.
output_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# Part 3: Define the ID of the image that we want to test.
# We already confirmed that this ID exists in the annotation folder
# and in the second training image folder.
# ============================================================

image_id = "aVuSPwmImhhe01o_hJm7IA"

# These are the possible image extensions that we will check.
possible_image_extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

# Put the three training image folders into a list.
training_image_folders = [
    train_image_folder_0,
    train_image_folder_1,
    train_image_folder_2
]

# Start with an empty value because we have not found the image yet.
image_path = None


# ============================================================
# Part 4: Search the three training folders for the original image.
# The program checks the image ID with different file extensions.
# ============================================================

for current_folder in training_image_folders:

    # Check whether the current folder exists.
    if current_folder.exists():

        # Try each possible image extension.
        for current_extension in possible_image_extensions:

            # Build the complete path of the possible image file.
            possible_image_path = (
                current_folder / (image_id + current_extension)
            )

            # If the image exists, remember its path.
            if possible_image_path.exists():

                image_path = possible_image_path

                # Stop checking extensions after finding the image.
                break

    # Stop searching folders after finding the image.
    if image_path is not None:
        break


# ============================================================
# Part 5: Find the JSON annotation file for this image.
# The annotation file should have the same image ID.
# ============================================================

annotation_path = annotation_folder / (image_id + ".json")


# ============================================================
# Part 6: Check whether the image and annotation files exist.
# If a file cannot be found, display a clear message.
# ============================================================

if image_path is None:
    print("ERROR: The original image could not be found.")
    print("Please check the three training image folder paths.")

elif not annotation_path.exists():
    print("ERROR: The JSON annotation file could not be found.")
    print("Please check the annotation folder path.")

else:
    print("The original image was found:")
    print(image_path)

    print("The annotation file was found:")
    print(annotation_path)


    # ========================================================
    # Part 7: Open the original image and read the JSON file.
    # We need both files to crop the correct traffic sign regions.
    # ========================================================

    original_image = Image.open(image_path).convert("RGB")

    with open(annotation_path, "r", encoding="utf-8") as json_file:
        annotation_data = json.load(json_file)

    # Get the actual image dimensions from the image file.
    actual_image_width, actual_image_height = original_image.size

    # Get the image dimensions recorded in the JSON annotation.
    json_image_width = annotation_data["width"]
    json_image_height = annotation_data["height"]

    print("Actual image width:", actual_image_width)
    print("Actual image height:", actual_image_height)

    print("JSON image width:", json_image_width)
    print("JSON image height:", json_image_height)


    # ========================================================
    # Part 8: Calculate the coordinate scaling factors.
    # This handles cases where the actual image size is different
    # from the dimensions recorded in the JSON file.
    # ========================================================

    scale_x = actual_image_width / json_image_width
    scale_y = actual_image_height / json_image_height

    print("Horizontal coordinate scale:", scale_x)
    print("Vertical coordinate scale:", scale_y)


    # ========================================================
    # Part 9: Read the traffic sign objects from the JSON file.
    # Each object contains a label and a bounding box.
    # ========================================================

    traffic_sign_objects = annotation_data["objects"]

    print("Number of annotated objects:", len(traffic_sign_objects))


    # ========================================================
    # Part 10: Crop every annotated object and save it separately.
    # The bounding box is converted to the actual image coordinates.
    # ========================================================

    saved_crop_count = 0

    for object_number, traffic_sign_object in enumerate(
        traffic_sign_objects,
        start=1
    ):

        # Read the traffic sign category.
        traffic_sign_label = traffic_sign_object["label"]

        # Read the bounding box coordinates from the JSON file.
        bounding_box = traffic_sign_object["bbox"]

        xmin = bounding_box["xmin"]
        ymin = bounding_box["ymin"]
        xmax = bounding_box["xmax"]
        ymax = bounding_box["ymax"]

        # Convert the JSON coordinates to the actual image coordinates.
        xmin = xmin * scale_x
        ymin = ymin * scale_y
        xmax = xmax * scale_x
        ymax = ymax * scale_y

        # Round the coordinates outward to avoid cutting off edge pixels.
        xmin = math.floor(xmin)
        ymin = math.floor(ymin)
        xmax = math.ceil(xmax)
        ymax = math.ceil(ymax)

        # Make sure that the coordinates stay inside the image.
        xmin = max(0, min(xmin, actual_image_width))
        ymin = max(0, min(ymin, actual_image_height))
        xmax = max(0, min(xmax, actual_image_width))
        ymax = max(0, min(ymax, actual_image_height))

        # Check whether the bounding box has a valid size.
        if xmax <= xmin or ymax <= ymin:
            print("Skipped an invalid bounding box:", traffic_sign_label)
            continue

        # Crop the traffic sign from the original street image.
        cropped_image = original_image.crop(
            (xmin, ymin, xmax, ymax)
        )

        # Replace characters that are unsuitable for Windows filenames.
        safe_label = traffic_sign_label.replace("/", "_")
        safe_label = safe_label.replace("\\", "_")
        safe_label = safe_label.replace(":", "_")

        # Build a filename containing the image ID and category.
        crop_filename = (
            image_id
            + "_object_"
            + str(object_number)
            + "_"
            + safe_label
            + ".jpg"
        )

        # Create the full output path.
        crop_output_path = output_folder / crop_filename

        # Save the cropped image as a JPEG file.
        cropped_image.save(crop_output_path, quality=95)

        # Increase the number of successfully saved crops.
        saved_crop_count = saved_crop_count + 1

        # Print information about the crop for debugging.
        print("----------------------------------------")
        print("Object number:", object_number)
        print("Traffic sign label:", traffic_sign_label)
        print("Crop coordinates:", xmin, ymin, xmax, ymax)
        print("Cropped image size:", cropped_image.size)
        print("Saved file:", crop_output_path)


    # ========================================================
    # Part 11: Print a final summary of the cropping test.
    # We will inspect these results before processing the full dataset.
    # ========================================================

    print("========================================")
    print("Cropping test finished.")
    print("Number of cropped objects:", saved_crop_count)
    print("Output folder:", output_folder)
    print("Please open the output folder and inspect the images.")

In [ ]:
# ============================================================
# Part 1: Import Path and Image.
# Path is used to check the file location.
# Image is used to open the JPG image.
# ============================================================

from pathlib import Path
from PIL import Image


# ============================================================
# Part 2: Set the exact image path copied from your computer.
# This path uses the real folder name "data2".
# ============================================================

image_path = Path(
    r"C:\Users\ASUS\Desktop\data2\images\aVuSPwmImhhe01o_hJm7IA.jpg"
)


# ============================================================
# Part 3: Check whether the image file exists.
# If the path is correct, Python should print True.
# ============================================================

print("Image path:")
print(image_path)

print("Does the image file exist?")
print(image_path.exists())


# ============================================================
# Part 4: Open the image and display its actual dimensions.
# This helps us confirm that Python can read the original JPG.
# ============================================================

if image_path.exists():

    original_image = Image.open(image_path)

    print("Image format:", original_image.format)
    print("Image dimensions:", original_image.size)

    original_image.show()

else:

    print("ERROR: Python still cannot find this image.")
    print("Please check the folder name and the file extension.")

In [ ]:
# ============================================================
# Part 1: Import the libraries.
# Path is used to work with file paths.
# json is used to read the annotation file.
# Image is used to open and crop the traffic sign images.
# ============================================================

from pathlib import Path
import json
import math
from PIL import Image


# ============================================================
# Part 2: Set the exact paths of the image and annotation.
# These paths are based on the locations confirmed on your computer.
# ============================================================

image_path = Path(
    r"C:\Users\ASUS\Desktop\data2\images\aVuSPwmImhhe01o_hJm7IA.jpg"
)

annotation_path = Path(
    r"C:\Users\ASUS\Desktop\Annotations\mtsd_v2_fully_annotated\annotations\aVuSPwmImhhe01o_hJm7IA.json"
)

output_folder = Path(
    r"C:\Users\ASUS\Desktop\Crop_Test"
)


# ============================================================
# Part 3: Create the output folder.
# If the folder already exists, the program will reuse it.
# ============================================================

output_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# Part 4: Check whether both input files exist.
# Stop the program if either file cannot be found.
# ============================================================

if not image_path.exists():
    raise FileNotFoundError(
        "The original JPG image cannot be found: "
        + str(image_path)
    )

if not annotation_path.exists():
    raise FileNotFoundError(
        "The JSON annotation file cannot be found: "
        + str(annotation_path)
    )


# ============================================================
# Part 5: Open the original JPG image.
# Read the actual image dimensions from the image file.
# ============================================================

original_image = Image.open(image_path).convert("RGB")

actual_image_width = original_image.width
actual_image_height = original_image.height

print("Original image was opened successfully.")
print("Actual image width:", actual_image_width)
print("Actual image height:", actual_image_height)


# ============================================================
# Part 6: Read the JSON annotation file.
# The JSON contains the image dimensions and annotated objects.
# ============================================================

with open(annotation_path, "r", encoding="utf-8") as json_file:
    annotation_data = json.load(json_file)

json_image_width = annotation_data["width"]
json_image_height = annotation_data["height"]

print("JSON image width:", json_image_width)
print("JSON image height:", json_image_height)


# ============================================================
# Part 7: Calculate the coordinate scaling factors.
# If the JSON and image dimensions match, both factors equal 1.
# ============================================================

scale_x = actual_image_width / json_image_width
scale_y = actual_image_height / json_image_height

print("Horizontal scaling factor:", scale_x)
print("Vertical scaling factor:", scale_y)


# ============================================================
# Part 8: Check whether the image dimensions match the JSON.
# The program can continue with coordinate scaling if necessary.
# ============================================================

if (
    actual_image_width == json_image_width
    and actual_image_height == json_image_height
):
    print("The image dimensions match the JSON annotation.")
else:
    print("The image dimensions do not match the JSON dimensions.")
    print("The bounding box coordinates will be scaled.")


# ============================================================
# Part 9: Get the list of annotated traffic sign objects.
# Each object contains a category label and a bounding box.
# ============================================================

traffic_sign_objects = annotation_data["objects"]

print("Number of annotated objects:", len(traffic_sign_objects))


# ============================================================
# Part 10: Crop every valid annotated object.
# Convert the bounding box coordinates to the actual image size.
# Save each cropped traffic sign as a separate JPG file.
# ============================================================

saved_crop_count = 0

for object_number, traffic_sign_object in enumerate(
    traffic_sign_objects,
    start=1
):

    # Read the category label of the current object.
    traffic_sign_label = traffic_sign_object["label"]

    # Read the bounding box coordinates from the JSON file.
    bounding_box = traffic_sign_object["bbox"]

    xmin = bounding_box["xmin"]
    ymin = bounding_box["ymin"]
    xmax = bounding_box["xmax"]
    ymax = bounding_box["ymax"]

    # Scale the coordinates to match the actual JPG image.
    xmin = xmin * scale_x
    ymin = ymin * scale_y
    xmax = xmax * scale_x
    ymax = ymax * scale_y

    # Round the coordinates outward to include the edge pixels.
    xmin = math.floor(xmin)
    ymin = math.floor(ymin)
    xmax = math.ceil(xmax)
    ymax = math.ceil(ymax)

    # Keep all coordinates inside the image boundaries.
    xmin = max(0, min(xmin, actual_image_width))
    ymin = max(0, min(ymin, actual_image_height))
    xmax = max(0, min(xmax, actual_image_width))
    ymax = max(0, min(ymax, actual_image_height))

    # Skip this object if its bounding box is invalid.
    if xmax <= xmin or ymax <= ymin:
        print("Skipping invalid bounding box:", traffic_sign_label)
        continue

    # Crop the current traffic sign from the original image.
    cropped_image = original_image.crop(
        (xmin, ymin, xmax, ymax)
    )

    # Replace special characters that may cause filename problems.
    safe_label = traffic_sign_label.replace("/", "_")
    safe_label = safe_label.replace("\\", "_")
    safe_label = safe_label.replace(":", "_")

    # Create a filename using the image ID, object number, and label.
    crop_filename = (
        "aVuSPwmImhhe01o_hJm7IA"
        + "_object_"
        + str(object_number)
        + "_"
        + safe_label
        + ".jpg"
    )

    # Set the full path for the cropped image.
    crop_output_path = output_folder / crop_filename

    # Save the cropped image to the output folder.
    cropped_image.save(crop_output_path, quality=95)

    # Count the successfully saved images.
    saved_crop_count = saved_crop_count + 1

    # Print the details of the current cropped traffic sign.
    print("----------------------------------------")
    print("Object number:", object_number)
    print("Traffic sign label:", traffic_sign_label)
    print("Bounding box:", xmin, ymin, xmax, ymax)
    print("Cropped image size:", cropped_image.size)
    print("Saved image:", crop_output_path)


# ============================================================
# Part 11: Display the final results.
# Check the output folder after the program finishes.
# ============================================================

print("========================================")
print("Cropping test finished.")
print("Number of cropped objects:", saved_crop_count)
print("Output folder:", output_folder)
print("Please open Crop_Test and inspect the cropped images.")

In [ ]:
# ============================================================
# Part 1: Import the libraries used for batch image processing.
# These libraries help us read JSON files, search for images,
# crop traffic signs, and save the results.
# ============================================================

import json
import math
from pathlib import Path
from PIL import Image


# ============================================================
# Part 2: Set the folders used in the current project.
# These paths follow the folder structure that we confirmed.
# ============================================================

desktop_path = Path(r"C:\Users\ASUS\Desktop")

training_image_folders = [
    desktop_path / "data2" / "images",
    desktop_path / "Data 2" / "images",
    desktop_path / "Data3" / "images",
    desktop_path / "data" / "images"
]

annotation_folder = (
    desktop_path
    / "Annotations"
    / "mtsd_v2_fully_annotated"
    / "annotations"
)

split_folder = (
    desktop_path
    / "Annotations"
    / "mtsd_v2_fully_annotated"
    / "splits"
)

output_folder = desktop_path / "MTSD_Cropped" / "train"

# Create the output folder before saving any cropped images.
output_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# Part 3: Load the training split file.
# Each line contains an image ID assigned to the training split.
# ============================================================

train_split_path = split_folder / "train"

if not train_split_path.exists():
    raise FileNotFoundError(
        "The train split file cannot be found: "
        + str(train_split_path)
    )

with open(train_split_path, "r", encoding="utf-8") as split_file:
    training_image_ids = set()

    for line in split_file:
        current_image_id = line.strip()

        if current_image_id != "":
            training_image_ids.add(current_image_id)

print("Number of image IDs in the train split:", len(training_image_ids))


# ============================================================
# Part 4: Build a dictionary of the available original images.
# The dictionary uses the image ID as the key and the file path
# as the value, so we can find images without searching repeatedly.
# ============================================================

image_path_dictionary = {}

for current_folder in training_image_folders:

    if current_folder.exists():

        print("Searching image folder:", current_folder)

        for current_image_path in current_folder.iterdir():

            if current_image_path.is_file():

                current_extension = current_image_path.suffix.lower()

                if current_extension in [".jpg", ".jpeg", ".png"]:

                    current_image_id = current_image_path.stem

                    if current_image_id not in image_path_dictionary:
                        image_path_dictionary[current_image_id] = current_image_path

    else:
        print("Folder not found, skipping:", current_folder)

print("Number of unique image IDs found:", len(image_path_dictionary))


# ============================================================
# Part 5: Create lists to record missing files and processing results.
# These records help us identify problems after the program finishes.
# ============================================================

missing_image_ids = []
missing_annotation_ids = []
failed_image_ids = []

saved_crop_count = 0
processed_image_count = 0


# ============================================================
# Part 6: Create a dictionary for mapping class labels to numbers.
# The class mapping will be saved after all available training images
# have been processed.
# ============================================================

class_to_index = {}


# ============================================================
# Part 7: Process every image ID in the training split.
# Read its JSON annotation and crop every annotated object.
# ============================================================

for image_number, current_image_id in enumerate(
    sorted(training_image_ids),
    start=1
):

    # Print progress so that we can see how much work is completed.
    if image_number % 100 == 0:
        print(
            "Processing image",
            image_number,
            "out of",
            len(training_image_ids)
        )

    # Find the original image using its image ID.
    if current_image_id not in image_path_dictionary:
        missing_image_ids.append(current_image_id)
        continue

    current_image_path = image_path_dictionary[current_image_id]

    # Find the JSON annotation file with the same image ID.
    current_annotation_path = (
        annotation_folder / (current_image_id + ".json")
    )

    if not current_annotation_path.exists():
        missing_annotation_ids.append(current_image_id)
        continue

    try:

        # Open the original street image.
        original_image = Image.open(current_image_path).convert("RGB")

        actual_image_width = original_image.width
        actual_image_height = original_image.height

        # Read the JSON annotation information.
        with open(
            current_annotation_path,
            "r",
            encoding="utf-8"
        ) as json_file:

            annotation_data = json.load(json_file)

        json_image_width = annotation_data["width"]
        json_image_height = annotation_data["height"]

        # Calculate the coordinate scale factors.
        # This preserves the original annotation locations if image
        # dimensions differ from the dimensions recorded in the JSON.
        scale_x = actual_image_width / json_image_width
        scale_y = actual_image_height / json_image_height

        # Read all annotated objects in the current image.
        traffic_sign_objects = annotation_data["objects"]

        # Crop every annotated object from the original image.
        for object_number, traffic_sign_object in enumerate(
            traffic_sign_objects,
            start=1
        ):

            # Read the original category label.
            traffic_sign_label = traffic_sign_object["label"]

            # Read the original bounding box coordinates.
            bounding_box = traffic_sign_object["bbox"]

            xmin = bounding_box["xmin"]
            ymin = bounding_box["ymin"]
            xmax = bounding_box["xmax"]
            ymax = bounding_box["ymax"]

            # Convert the coordinates to the actual image dimensions.
            xmin = xmin * scale_x
            ymin = ymin * scale_y
            xmax = xmax * scale_x
            ymax = ymax * scale_y

            # Round the coordinates outward to include boundary pixels.
            xmin = math.floor(xmin)
            ymin = math.floor(ymin)
            xmax = math.ceil(xmax)
            ymax = math.ceil(ymax)

            # Keep the bounding box inside the image.
            xmin = max(0, min(xmin, actual_image_width))
            ymin = max(0, min(ymin, actual_image_height))
            xmax = max(0, min(xmax, actual_image_width))
            ymax = max(0, min(ymax, actual_image_height))

            # Skip bounding boxes with zero or negative dimensions.
            if xmax <= xmin or ymax <= ymin:
                continue

            # Crop the original bounding box without extra padding.
            cropped_image = original_image.crop(
                (xmin, ymin, xmax, ymax)
            )

            # Assign a number to each category label.
            if traffic_sign_label not in class_to_index:
                class_to_index[traffic_sign_label] = len(class_to_index)

            class_index = class_to_index[traffic_sign_label]

            # Make a safe filename for Windows.
            safe_label = traffic_sign_label.replace("/", "_")
            safe_label = safe_label.replace("\\", "_")
            safe_label = safe_label.replace(":", "_")

            # Include the image ID and object number to avoid filename conflicts.
            crop_filename = (
                current_image_id
                + "_object_"
                + str(object_number)
                + "_"
                + safe_label
                + ".jpg"
            )

            # Save the cropped traffic sign image.
            crop_output_path = output_folder / crop_filename

            cropped_image.save(
                crop_output_path,
                quality=95
            )

            saved_crop_count = saved_crop_count + 1

        processed_image_count = processed_image_count + 1

    except Exception as error:

        # Record images that could not be processed.
        failed_image_ids.append(
            {
                "image_id": current_image_id,
                "error": str(error)
            }
        )


# ============================================================
# Part 8: Save the class mapping.
# This JSON file records the relationship between category names
# and their numeric class IDs.
# ============================================================

class_mapping_path = output_folder / "class_to_index.json"

with open(
    class_mapping_path,
    "w",
    encoding="utf-8"
) as mapping_file:

    json.dump(
        class_to_index,
        mapping_file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# Part 9: Save the processing report.
# The report records the number of processed images, saved crops,
# and any missing or failed files.
# ============================================================

processing_report = {
    "training_split_image_count": len(training_image_ids),
    "processed_image_count": processed_image_count,
    "saved_crop_count": saved_crop_count,
    "missing_image_count": len(missing_image_ids),
    "missing_annotation_count": len(missing_annotation_ids),
    "failed_image_count": len(failed_image_ids),
    "missing_image_ids": missing_image_ids,
    "missing_annotation_ids": missing_annotation_ids,
    "failed_images": failed_image_ids
}

report_path = output_folder / "processing_report.json"

with open(
    report_path,
    "w",
    encoding="utf-8"
) as report_file:

    json.dump(
        processing_report,
        report_file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# Part 10: Print the final processing summary.
# Review these numbers before continuing to validation and testing.
# ============================================================

print("========================================")
print("Batch cropping finished.")
print("Training IDs in split:", len(training_image_ids))
print("Successfully processed images:", processed_image_count)
print("Saved cropped traffic signs:", saved_crop_count)
print("Missing images:", len(missing_image_ids))
print("Missing annotations:", len(missing_annotation_ids))
print("Failed images:", len(failed_image_ids))
print("Class mapping saved to:", class_mapping_path)
print("Processing report saved to:", report_path)
print("Output folder:", output_folder)

In [ ]:
# ============================================================
# Part 1: Import Path to inspect the dataset folder structure.
# This helps us find the actual location of the train split file.
# ============================================================

from pathlib import Path


# ============================================================
# Part 2: Set the expected annotation folder path.
# This is the path that the previous program tried to use.
# ============================================================

desktop_path = Path(r"C:\Users\ASUS\Desktop")

annotation_main_folder = (
    desktop_path
    / "Annotations"
    / "mtsd_v2_fully_annotated"
)

split_folder = annotation_main_folder / "splits"


# ============================================================
# Part 3: Check whether the main folders exist.
# Print the result for each folder to identify the problem.
# ============================================================

print("Desktop exists:", desktop_path.exists())

print("Annotation folder exists:", annotation_main_folder.exists())

print("Split folder exists:", split_folder.exists())


# ============================================================
# Part 4: Display the contents of the annotation folder.
# This shows the actual names of the files and subfolders.
# ============================================================

if annotation_main_folder.exists():

    print("\nContents of the annotation folder:")

    for item in annotation_main_folder.iterdir():
        print(item.name)


# ============================================================
# Part 5: Display the contents of the split folder.
# This checks whether train, val, and test are present.
# ============================================================

if split_folder.exists():

    print("\nContents of the split folder:")

    for item in split_folder.iterdir():
        print(item.name)

else:

    print("\nThe expected split folder does not exist.")

In [ ]:
# ============================================================
# Part 1: Import the libraries.
# These libraries are used to read files, process JSON data,
# crop traffic signs, and save the results.
# ============================================================

import json
import math
from pathlib import Path
from PIL import Image


# ============================================================
# Part 2: Set the dataset folder paths.
# These paths are based on the actual folders on your computer.
# ============================================================

desktop_path = Path(r"C:\Users\ASUS\Desktop")

# List the possible locations of the downloaded training images.
training_image_folders = [
    desktop_path / "data" / "images",
    desktop_path / "data2" / "images",
    desktop_path / "Data 2" / "images",
    desktop_path / "Data3" / "images"
]

# Set the folder containing the JSON annotation files.
annotation_folder = (
    desktop_path
    / "Annotations"
    / "mtsd_v2_fully_annotated"
    / "annotations"
)

# Set the folder containing the train, val, and test split files.
split_folder = (
    desktop_path
    / "Annotations"
    / "mtsd_v2_fully_annotated"
    / "splits"
)

# Set the output folder for the cropped training images.
output_folder = desktop_path / "MTSD_Cropped" / "train"

# Create the output folder if it does not already exist.
output_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# Part 3: Check the important input folders and files.
# Stop the program early if a required path is incorrect.
# ============================================================

if not annotation_folder.exists():
    raise FileNotFoundError(
        "The annotation folder cannot be found: "
        + str(annotation_folder)
    )

if not split_folder.exists():
    raise FileNotFoundError(
        "The split folder cannot be found: "
        + str(split_folder)
    )

train_split_path = split_folder / "train.txt"

if not train_split_path.exists():
    raise FileNotFoundError(
        "The training split file cannot be found: "
        + str(train_split_path)
    )

print("Annotation folder:", annotation_folder)
print("Training split file:", train_split_path)
print("Output folder:", output_folder)


# ============================================================
# Part 4: Read the training image IDs from train.txt.
# Each non-empty line contains an image ID in the training split.
# ============================================================

training_image_ids = set()

with open(train_split_path, "r", encoding="utf-8") as split_file:

    for line in split_file:

        current_image_id = line.strip()

        if current_image_id != "":
            training_image_ids.add(current_image_id)

print("Number of training IDs:", len(training_image_ids))


# ============================================================
# Part 5: Build a dictionary of the available image files.
# The image ID is used as the dictionary key.
# The complete image path is used as the dictionary value.
# ============================================================

image_path_dictionary = {}

for current_folder in training_image_folders:

    if current_folder.exists():

        print("Searching folder:", current_folder)

        for current_image_path in current_folder.iterdir():

            if current_image_path.is_file():

                current_extension = current_image_path.suffix.lower()

                if current_extension in [".jpg", ".jpeg", ".png"]:

                    current_image_id = current_image_path.stem

                    # Keep the first path found for each image ID.
                    if current_image_id not in image_path_dictionary:
                        image_path_dictionary[current_image_id] = (
                            current_image_path
                        )

    else:
        print("Folder does not exist, skipping:", current_folder)

print("Number of unique images found:", len(image_path_dictionary))


# ============================================================
# Part 6: Prepare the processing statistics.
# These lists record missing images, missing annotations,
# and files that could not be processed.
# ============================================================

missing_image_ids = []
missing_annotation_ids = []
failed_images = []

processed_image_count = 0
saved_crop_count = 0

# This dictionary maps each category label to a numeric class ID.
class_to_index = {}


# ============================================================
# Part 7: Process every image listed in train.txt.
# Each image is matched with its JSON annotation file.
# ============================================================

for image_number, current_image_id in enumerate(
    sorted(training_image_ids),
    start=1
):

    # Print progress every 100 image IDs.
    if image_number % 100 == 0:
        print(
            "Processing image",
            image_number,
            "of",
            len(training_image_ids)
        )

    # Find the original image using its image ID.
    if current_image_id not in image_path_dictionary:

        missing_image_ids.append(current_image_id)
        continue

    current_image_path = image_path_dictionary[current_image_id]

    # Find the JSON annotation file with the same image ID.
    current_annotation_path = (
        annotation_folder / (current_image_id + ".json")
    )

    if not current_annotation_path.exists():

        missing_annotation_ids.append(current_image_id)
        continue

    try:

        # Open the original street image.
        original_image = Image.open(current_image_path).convert("RGB")

        actual_image_width = original_image.width
        actual_image_height = original_image.height

        # Read the JSON annotation data.
        with open(
            current_annotation_path,
            "r",
            encoding="utf-8"
        ) as json_file:

            annotation_data = json.load(json_file)

        # Read the image dimensions recorded in the JSON file.
        json_image_width = annotation_data["width"]
        json_image_height = annotation_data["height"]

        # Calculate coordinate scale factors.
        # If the image dimensions match the JSON, both factors equal 1.
        scale_x = actual_image_width / json_image_width
        scale_y = actual_image_height / json_image_height

        # Read all annotated objects in the current image.
        traffic_sign_objects = annotation_data["objects"]

        # Process every annotated object separately.
        for object_number, traffic_sign_object in enumerate(
            traffic_sign_objects,
            start=1
        ):

            # Read the original traffic sign category.
            traffic_sign_label = traffic_sign_object["label"]

            # Read the original bounding box coordinates.
            bounding_box = traffic_sign_object["bbox"]

            xmin = bounding_box["xmin"]
            ymin = bounding_box["ymin"]
            xmax = bounding_box["xmax"]
            ymax = bounding_box["ymax"]

            # Convert the coordinates to the actual image dimensions.
            xmin = xmin * scale_x
            ymin = ymin * scale_y
            xmax = xmax * scale_x
            ymax = ymax * scale_y

            # Round the coordinates to integer pixel positions.
            xmin = math.floor(xmin)
            ymin = math.floor(ymin)
            xmax = math.ceil(xmax)
            ymax = math.ceil(ymax)

            # Keep the coordinates inside the image boundaries.
            xmin = max(0, min(xmin, actual_image_width))
            ymin = max(0, min(ymin, actual_image_height))
            xmax = max(0, min(xmax, actual_image_width))
            ymax = max(0, min(ymax, actual_image_height))

            # Skip invalid bounding boxes.
            if xmax <= xmin or ymax <= ymin:
                continue

            # Crop the original bounding box without extra padding.
            cropped_image = original_image.crop(
                (xmin, ymin, xmax, ymax)
            )

            # Assign a numeric ID to each category.
            if traffic_sign_label not in class_to_index:

                class_to_index[traffic_sign_label] = len(class_to_index)

            # Replace characters that may cause Windows filename problems.
            safe_label = traffic_sign_label.replace("/", "_")
            safe_label = safe_label.replace("\\", "_")
            safe_label = safe_label.replace(":", "_")

            # Build a unique filename for the cropped object.
            crop_filename = (
                current_image_id
                + "_object_"
                + str(object_number)
                + "_"
                + safe_label
                + ".jpg"
            )

            # Set the full path of the cropped image.
            crop_output_path = output_folder / crop_filename

            # Save the cropped traffic sign as a JPEG image.
            cropped_image.save(
                crop_output_path,
                quality=95
            )

            # Increase the total number of saved cropped images.
            saved_crop_count += 1

        # Count the image as processed after its objects are handled.
        processed_image_count += 1

    except Exception as error:

        # Record the ID and error message if processing fails.
        failed_images.append(
            {
                "image_id": current_image_id,
                "error": str(error)
            }
        )


# ============================================================
# Part 8: Save the category-to-number mapping.
# This file records which numeric ID belongs to each category.
# We will later use one fixed mapping for all dataset splits.
# ============================================================

class_mapping_path = output_folder / "class_to_index.json"

with open(
    class_mapping_path,
    "w",
    encoding="utf-8"
) as mapping_file:

    json.dump(
        class_to_index,
        mapping_file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# Part 9: Save the processing report.
# The report includes successful processing counts and missing files.
# ============================================================

processing_report = {
    "training_split_image_count": len(training_image_ids),
    "available_image_count": len(image_path_dictionary),
    "processed_image_count": processed_image_count,
    "saved_crop_count": saved_crop_count,
    "missing_image_count": len(missing_image_ids),
    "missing_annotation_count": len(missing_annotation_ids),
    "failed_image_count": len(failed_images),
    "missing_image_ids": missing_image_ids,
    "missing_annotation_ids": missing_annotation_ids,
    "failed_images": failed_images
}

report_path = output_folder / "processing_report.json"

with open(
    report_path,
    "w",
    encoding="utf-8"
) as report_file:

    json.dump(
        processing_report,
        report_file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# Part 10: Print the final processing summary.
# Use these results to check the dataset before training the CNN.
# ============================================================

print("========================================")
print("Batch cropping finished.")
print("Training IDs:", len(training_image_ids))
print("Available image IDs:", len(image_path_dictionary))
print("Processed images:", processed_image_count)
print("Saved cropped objects:", saved_crop_count)
print("Missing images:", len(missing_image_ids))
print("Missing annotations:", len(missing_annotation_ids))
print("Failed images:", len(failed_images))
print("Class mapping saved to:", class_mapping_path)
print("Processing report saved to:", report_path)
print("Output folder:", output_folder)

In [ ]:
# This code counts how many cropped images belong to each traffic sign class.
# It also saves the counting results into a CSV file for later analysis.

import os
import json
import pandas as pd

# Step 1: Set the folder paths.
# This is the folder containing all the cropped training images.
train_folder = r".\train"

# This file contains the mapping between traffic sign class names and numbers.
class_mapping_path = os.path.join(train_folder, "class_to_index.json")

# Step 2: Load the class mapping file.
# The dictionary uses the traffic sign class name as the key
# and the corresponding numerical label as the value.
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

# Step 3: Create an empty dictionary to store the image counts.
# Each class will have its own count.
class_image_counts = {}

for class_name in class_to_index.keys():
    class_image_counts[class_name] = 0

# Step 4: Go through all class folders and count the image files.
# The class folders are expected to be inside the training folder.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for class_name in class_to_index.keys():

    # Create the path for the current class folder.
    class_folder = os.path.join(train_folder, class_name)

    # Check whether the class folder exists.
    if os.path.isdir(class_folder):

        # Get all files in the current class folder.
        file_names = os.listdir(class_folder)

        # Count the files that have supported image extensions.
        for file_name in file_names:
            if file_name.lower().endswith(image_extensions):
                class_image_counts[class_name] += 1

# Step 5: Convert the counting dictionary into a table.
# The table will contain the class name, class number, and image count.
results = []

for class_name, class_number in class_to_index.items():

    image_count = class_image_counts[class_name]

    results.append({
        "class_name": class_name,
        "class_number": class_number,
        "image_count": image_count
    })

results_df = pd.DataFrame(results)

# Step 6: Sort the table by image count, from the largest to the smallest.
results_df = results_df.sort_values(
    by="image_count",
    ascending=False
)

# Step 7: Print some summary information.
# This helps us understand the overall distribution of the dataset.
print("Number of classes:", len(class_to_index))
print("Total cropped images counted:", results_df["image_count"].sum())
print("Classes with no images:", (results_df["image_count"] == 0).sum())
print("Largest class image count:", results_df["image_count"].max())
print("Smallest class image count:", results_df["image_count"].min())

# Step 8: Display the 20 classes with the most images.
print("\nTop 20 classes with the most images:")
print(results_df.head(20).to_string(index=False))

# Step 9: Display the 20 classes with the fewest images.
print("\nTop 20 classes with the fewest images:")
print(results_df.tail(20).sort_values(
    by="image_count",
    ascending=True
).to_string(index=False))

# Step 10: Save the complete counting results as a CSV file.
# The CSV file can be opened using Excel.
output_csv_path = os.path.join(
    train_folder,
    "class_distribution.csv"
)

results_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

print("\nClass distribution results saved to:")
print(output_csv_path)

In [ ]:
# This code checks the actual structure of the cropped training dataset.
# It helps us find where the cropped images are stored.

import os

# Step 1: Set the path of the training output folder.
train_folder = r".\train"

# Step 2: Check whether the training folder exists.
print("Training folder exists:", os.path.exists(train_folder))
print("Training folder path:", train_folder)

# Step 3: List the first 20 items in the training folder.
# These items can be image files or subfolders.
all_items = os.listdir(train_folder)

print("\nNumber of items directly inside train:", len(all_items))
print("\nFirst 20 items:")

for item in all_items[:20]:
    item_path = os.path.join(train_folder, item)

    if os.path.isdir(item_path):
        print("[FOLDER]", item)
    else:
        print("[FILE]", item)

# Step 4: Count image files directly inside the training folder.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

direct_image_count = 0

for item in all_items:
    item_path = os.path.join(train_folder, item)

    if os.path.isfile(item_path):
        if item.lower().endswith(image_extensions):
            direct_image_count += 1

print("\nImages directly inside train:", direct_image_count)

# Step 5: Count image files inside all subfolders.
# This checks whether the images were saved in nested folders.
nested_image_count = 0
folder_count = 0

for current_folder, subfolders, file_names in os.walk(train_folder):

    for subfolder in subfolders:
        folder_count += 1

    for file_name in file_names:
        if file_name.lower().endswith(image_extensions):
            nested_image_count += 1

print("Number of subfolders found:", folder_count)
print("Total images found inside train and its subfolders:", nested_image_count)

In [ ]:
# This code counts the number of cropped images for each traffic sign class.
# The cropped images are stored directly inside the train folder.
# The traffic sign class is included at the end of each image filename.

import os
import json
import pandas as pd

# Step 1: Set the path to the folder containing the cropped training images.
train_folder = r".\train"

# Step 2: Set the path to the class mapping JSON file.
class_mapping_path = os.path.join(
    train_folder,
    "class_to_index.json"
)

# Step 3: Load the class names and their numerical labels.
# We use the existing mapping so that the class numbers stay unchanged.
with open(class_mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

# Step 4: Create a dictionary to store the image count for each class.
# At the beginning, every class has zero images.
class_image_counts = {}

for class_name in class_to_index:
    class_image_counts[class_name] = 0

# Step 5: Get all supported image files directly inside the train folder.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

all_files = os.listdir(train_folder)

# Step 6: Check every image filename and extract its class name.
# The filename format is:
# imageID_object_number_class_name.jpg
for file_name in all_files:

    # Check whether the current file is an image.
    if file_name.lower().endswith(image_extensions):

        # Remove the image extension from the filename.
        file_name_without_extension = os.path.splitext(file_name)[0]

        # Find the position of the object number separator.
        # The filename contains a section such as "_object_1_".
        object_position = file_name_without_extension.find("_object_")

        if object_position != -1:

            # Find the next underscore after the object number.
            # This separates the object number from the class name.
            class_start_position = file_name_without_extension.find(
                "_",
                object_position + len("_object_")
            )

            if class_start_position != -1:

                # Extract the class name from the filename.
                class_name = file_name_without_extension[
                    class_start_position + 1:
                ]

                # Add one to the count if the class exists in our mapping.
                if class_name in class_image_counts:
                    class_image_counts[class_name] += 1

# Step 7: Create a table containing class names, class numbers, and counts.
results = []

for class_name, class_number in class_to_index.items():

    results.append({
        "class_name": class_name,
        "class_number": class_number,
        "image_count": class_image_counts[class_name]
    })

results_df = pd.DataFrame(results)

# Step 8: Sort the results from the largest class to the smallest class.
results_df = results_df.sort_values(
    by="image_count",
    ascending=False
)

# Step 9: Print the main statistics.
print("Number of classes:", len(class_to_index))
print("Total cropped images counted:", results_df["image_count"].sum())
print("Classes with no images:", (results_df["image_count"] == 0).sum())
print("Largest class image count:", results_df["image_count"].max())

# Find the smallest non-zero class count.
non_zero_counts = results_df.loc[
    results_df["image_count"] > 0,
    "image_count"
]

if len(non_zero_counts) > 0:
    print("Smallest non-zero class image count:", non_zero_counts.min())
else:
    print("No images were matched to the class mapping.")

# Step 10: Display the 20 classes with the most images.
print("\nTop 20 classes with the most images:")
print(results_df.head(20).to_string(index=False))

# Step 11: Display the 20 classes with the fewest images.
print("\nTop 20 classes with the fewest images:")
print(
    results_df[results_df["image_count"] > 0]
    .sort_values(by="image_count", ascending=True)
    .head(20)
    .to_string(index=False)
)

# Step 12: Save the full class distribution table as a CSV file.
# This file can be opened in Excel.
output_csv_path = os.path.join(
    train_folder,
    "class_distribution.csv"
)

results_df.to_csv(
    output_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nClass distribution saved to:")
print(output_csv_path)

In [ ]:
# This code checks the dataset distribution without the other-sign class.
# It does not delete, move, or modify any cropped images.

import os
import pandas as pd

# Step 1: Set the path of the existing class distribution CSV file.
train_folder = r".\train"

distribution_path = os.path.join(
    train_folder,
    "class_distribution.csv"
)

# Step 2: Load the class distribution table.
class_distribution = pd.read_csv(distribution_path)

# Step 3: Remove other-sign only from this analysis table.
# The original image files will remain unchanged.
specific_sign_classes = class_distribution[
    class_distribution["class_name"] != "other-sign"
].copy()

# Step 4: Calculate the total number of images in the remaining classes.
total_specific_sign_images = specific_sign_classes["image_count"].sum()

# Step 5: Count classes with fewer than 30 images.
# We are only measuring the number here, not deleting these classes.
classes_below_30 = specific_sign_classes[
    specific_sign_classes["image_count"] < 30
]

# Step 6: Count classes with fewer than 10 images.
classes_below_10 = specific_sign_classes[
    specific_sign_classes["image_count"] < 10
]

# Step 7: Print the summary results.
print("Number of specific sign classes:", len(specific_sign_classes))
print("Total images excluding other-sign:", total_specific_sign_images)

print("Classes with fewer than 30 images:", len(classes_below_30))
print("Classes with fewer than 10 images:", len(classes_below_10))

print("\nClasses with fewer than 30 images:")
print(
    classes_below_30.sort_values(
        by="image_count",
        ascending=True
    ).to_string(index=False)
)

# Step 8: Save this analysis as a separate CSV file.
# This creates a new report and does not change the original distribution file.
output_path = os.path.join(
    train_folder,
    "class_distribution_without_other_sign.csv"
)

specific_sign_classes.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nAnalysis saved to:")
print(output_path)

In [ ]:
# This code creates the final list of traffic sign classes for our CNN project.
# It excludes the other-sign class and classes with fewer than 30 training images.
# It does not delete or modify any cropped images.

import os
import json
import pandas as pd

# Step 1: Set the folder containing the cropped training dataset.
train_folder = r".\train"

# Step 2: Set the path of the class distribution CSV file.
distribution_path = os.path.join(
    train_folder,
    "class_distribution.csv"
)

# Step 3: Load the class distribution table.
class_distribution = pd.read_csv(distribution_path)

# Step 4: Exclude the other-sign class.
# We want the model to classify specific traffic sign types.
specific_sign_classes = class_distribution[
    class_distribution["class_name"] != "other-sign"
].copy()

# Step 5: Keep classes with at least 30 training images.
# This rule helps reduce the problem caused by classes with very few samples.
selected_classes = specific_sign_classes[
    specific_sign_classes["image_count"] >= 30
].copy()

# Step 6: Sort the selected classes by their original class numbers.
# This keeps the list organized and easy to check.
selected_classes = selected_classes.sort_values(
    by="class_number",
    ascending=True
).reset_index(drop=True)

# Step 7: Create a new continuous class mapping.
# The new class numbers start at zero and follow the selected class list.
final_class_to_index = {}

for new_index in range(len(selected_classes)):
    class_name = selected_classes.loc[new_index, "class_name"]
    final_class_to_index[class_name] = new_index

# Step 8: Print the dataset selection summary.
print("Original number of classes:", len(class_distribution))
print("Classes excluding other-sign:", len(specific_sign_classes))
print("Classes kept after applying the 30-image rule:", len(selected_classes))
print("Total training images in the selected classes:",
      selected_classes["image_count"].sum())

# Step 9: Save the selected class list as a CSV file.
# This file records the class name, original number, and training image count.
selected_classes_csv_path = os.path.join(
    train_folder,
    "selected_classes.csv"
)

selected_classes.to_csv(
    selected_classes_csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Step 10: Save the new class mapping as a JSON file.
# We will reuse this mapping when preparing the validation dataset.
final_mapping_path = os.path.join(
    train_folder,
    "final_class_to_index.json"
)

with open(final_mapping_path, "w", encoding="utf-8") as file:
    json.dump(
        final_class_to_index,
        file,
        indent=4,
        ensure_ascii=False
    )

# Step 11: Print the output file locations.
print("\nSelected class list saved to:")
print(selected_classes_csv_path)

print("\nFinal class mapping saved to:")
print(final_mapping_path)

In [ ]:
# This code crops traffic signs from the validation images.
# It uses the original bounding boxes from the MTSD annotation files.
# It keeps only the 326 classes selected from the training dataset.
# It uses the existing final class mapping to keep labels consistent.
# It does not modify the original validation images or training images.

import os
import json
from PIL import Image

# Step 1: Set the paths for the validation images, annotations, and split file.
desktop_path = r"C:\Users\ASUS\Desktop"

validation_image_folder = os.path.join(
    desktop_path,
    "Data Val",
    "images"
)

annotation_folder = os.path.join(
    desktop_path,
    "Annotations",
    "mtsd_v2_fully_annotated",
    "annotations"
)

validation_split_path = os.path.join(
    desktop_path,
    "Annotations",
    "mtsd_v2_fully_annotated",
    "splits",
    "val.txt"
)

# Step 2: Set the output folder for the cropped validation images.
# The new folder is separate from the original training dataset.
validation_output_folder = os.path.join(
    desktop_path,
    "MTSD_Cropped",
    "val"
)

os.makedirs(
    validation_output_folder,
    exist_ok=True
)

# Step 3: Load the final class mapping created from the training dataset.
# We must use this mapping instead of creating new class numbers.
final_mapping_path = os.path.join(
    desktop_path,
    "MTSD_Cropped",
    "train",
    "final_class_to_index.json"
)

with open(final_mapping_path, "r", encoding="utf-8") as file:
    final_class_to_index = json.load(file)

# Step 4: Read the validation image IDs from val.txt.
# Each line contains an image ID without the .jpg extension.
with open(validation_split_path, "r", encoding="utf-8") as file:
    validation_image_ids = [
        line.strip()
        for line in file
        if line.strip()
    ]

print("Number of validation IDs:", len(validation_image_ids))
print("Number of selected classes:", len(final_class_to_index))

# Step 5: Define the supported image extensions.
# The original images in this dataset are expected to be JPEG files.
image_extensions = [".jpg", ".jpeg", ".png"]

# Step 6: Prepare counters to record the processing results.
processed_images = 0
saved_crops = 0
missing_images = 0
missing_annotations = 0
failed_images = 0
skipped_objects = 0

# Step 7: Process each validation image and its annotation.
for image_number, image_id in enumerate(validation_image_ids, start=1):

    # Print the progress every 100 images.
    if image_number % 100 == 0:
        print("Processing validation image", image_number,
              "of", len(validation_image_ids))

    # Step 7.1: Find the original validation image.
    image_path = None

    for extension in image_extensions:
        possible_image_path = os.path.join(
            validation_image_folder,
            image_id + extension
        )

        if os.path.isfile(possible_image_path):
            image_path = possible_image_path
            break

    # Skip the image if the original file cannot be found.
    if image_path is None:
        missing_images += 1
        continue

    # Step 7.2: Find the corresponding JSON annotation file.
    annotation_path = os.path.join(
        annotation_folder,
        image_id + ".json"
    )

    if not os.path.isfile(annotation_path):
        missing_annotations += 1
        continue

    try:
        # Step 7.3: Open the original image.
        image = Image.open(image_path).convert("RGB")

        # Step 7.4: Read the JSON annotation.
        with open(annotation_path, "r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        # Step 7.5: Read the original image dimensions from the annotation.
        annotation_width = annotation_data.get("width", image.width)
        annotation_height = annotation_data.get("height", image.height)

        # Step 7.6: Calculate scale factors if the dimensions differ.
        # Normally these factors should be 1 when the sizes match.
        scale_x = image.width / annotation_width
        scale_y = image.height / annotation_height

        # Step 7.7: Process every annotated traffic sign object.
        objects = annotation_data.get("objects", [])

        object_number = 0

        for object_data in objects:

            # Read the original traffic sign class name.
            class_name = object_data.get("label")

            # Keep only the classes selected from the training dataset.
            if class_name not in final_class_to_index:
                skipped_objects += 1
                continue

            # Read the original bounding box.
            bounding_box = object_data.get("bbox")

            if not bounding_box:
                skipped_objects += 1
                continue

            # Convert the bounding box coordinates into image coordinates.
            xmin = int(bounding_box["xmin"] * scale_x)
            ymin = int(bounding_box["ymin"] * scale_y)
            xmax = int(bounding_box["xmax"] * scale_x)
            ymax = int(bounding_box["ymax"] * scale_y)

            # Keep the coordinates inside the image boundaries.
            xmin = max(0, min(xmin, image.width))
            ymin = max(0, min(ymin, image.height))
            xmax = max(0, min(xmax, image.width))
            ymax = max(0, min(ymax, image.height))

            # Skip bounding boxes with invalid dimensions.
            if xmax <= xmin or ymax <= ymin:
                skipped_objects += 1
                continue

            # Crop the traffic sign using the original bounding box.
            cropped_image = image.crop(
                (xmin, ymin, xmax, ymax)
            )

            # Create a filename that includes the image ID,
            # object number, and traffic sign class.
            object_number += 1

            output_filename = (
                image_id
                + "_object_"
                + str(object_number)
                + "_"
                + class_name
                + ".jpg"
            )

            output_path = os.path.join(
                validation_output_folder,
                output_filename
            )

            # Save the cropped traffic sign.
            cropped_image.save(
                output_path,
                format="JPEG",
                quality=95
            )

            saved_crops += 1

        processed_images += 1

    except Exception as error:
        failed_images += 1
        print("Failed to process image:", image_id)
        print("Error:", error)

# Step 8: Print the final processing summary.
print("\n========================================")
print("Validation cropping finished.")
print("Validation IDs:", len(validation_image_ids))
print("Processed images:", processed_images)
print("Saved cropped objects:", saved_crops)
print("Skipped objects:", skipped_objects)
print("Missing images:", missing_images)
print("Missing annotations:", missing_annotations)
print("Failed images:", failed_images)
print("Output folder:", validation_output_folder)

# Step 9: Save a report for later checking.
# The report records the validation processing results.
validation_report = {
    "validation_split_image_count": len(validation_image_ids),
    "processed_image_count": processed_images,
    "saved_crop_count": saved_crops,
    "skipped_object_count": skipped_objects,
    "missing_image_count": missing_images,
    "missing_annotation_count": missing_annotations,
    "failed_image_count": failed_images,
    "selected_class_count": len(final_class_to_index)
}

report_path = os.path.join(
    validation_output_folder,
    "validation_processing_report.json"
)

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(
        validation_report,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Processing report saved to:", report_path)

In [ ]:
# This code checks the training and validation datasets before CNN training.
# It verifies the class mapping, counts the cropped images, and checks labels.
# It does not modify or delete any files.

import os
import json
import pandas as pd

# Step 1: Set the paths for the training and validation folders.
training_folder = r".\train"
validation_folder = r".\val"

# Step 2: Load the final class mapping used by the training dataset.
training_mapping_path = os.path.join(
    training_folder,
    "final_class_to_index.json"
)

with open(training_mapping_path, "r", encoding="utf-8") as file:
    training_class_mapping = json.load(file)

# Step 3: Load the same mapping file for comparison.
# The validation dataset should use exactly the same class mapping.
validation_class_mapping_path = os.path.join(
    training_folder,
    "final_class_to_index.json"
)

with open(validation_class_mapping_path, "r", encoding="utf-8") as file:
    validation_class_mapping = json.load(file)

# Step 4: Compare the two mappings.
# This confirms that every class has the same numerical label.
mappings_are_identical = (
    training_class_mapping == validation_class_mapping
)

print("Training and validation class mappings are identical:",
      mappings_are_identical)

print("Number of training classes:", len(training_class_mapping))
print("Number of validation classes:", len(validation_class_mapping))

# Step 5: Count the cropped images in both folders.
# The cropped images are stored directly inside each folder.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

training_image_count = 0

for file_name in os.listdir(training_folder):
    if file_name.lower().endswith(image_extensions):
        training_image_count += 1

validation_image_count = 0

for file_name in os.listdir(validation_folder):
    if file_name.lower().endswith(image_extensions):
        validation_image_count += 1

print("Training cropped image count:", training_image_count)
print("Validation cropped image count:", validation_image_count)

# Step 6: Load the saved validation processing report.
validation_report_path = os.path.join(
    validation_folder,
    "validation_processing_report.json"
)

with open(validation_report_path, "r", encoding="utf-8") as file:
    validation_report = json.load(file)

print("\nValidation processing report:")
for report_name, report_value in validation_report.items():
    print(report_name, ":", report_value)

# Step 7: Check whether the expected dataset sizes are correct.
print("\nDataset checks:")

if mappings_are_identical:
    print("PASS: Class mappings are consistent.")
else:
    print("WARNING: Class mappings do not match.")

if training_image_count == 60162:
    print("PASS: Training image count matches the selected dataset.")
else:
    print("WARNING: Training image count differs from the expected count.")

if validation_image_count == 8736:
    print("PASS: Validation image count matches the processing report.")
else:
    print("WARNING: Validation image count differs from the expected count.")

In [ ]:
# This code creates training and validation manifests for the CNN project.
# It keeps only the selected 326 classes.
# It does not copy, move, or delete any image files.
# Both datasets use the same final class-to-index mapping.

import os
import json
import pandas as pd

# Step 1: Set the paths for the training and validation folders.
training_folder = r".\train"
validation_folder = r".\val"

# Step 2: Load the final class mapping.
# This mapping contains the 326 classes selected from the training data.
mapping_path = os.path.join(
    training_folder,
    "final_class_to_index.json"
)

with open(mapping_path, "r", encoding="utf-8") as file:
    class_to_index = json.load(file)

# Step 3: Create a set of valid class names.
# A set makes it easier to check whether a filename belongs to a selected class.
selected_class_names = set(class_to_index.keys())

# Step 4: Define the supported image extensions.
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Step 5: Define a function to build a manifest from a folder.
# The function reads the class name from the end of each image filename.
def create_manifest(image_folder, selected_classes, class_mapping):

    manifest_rows = []

    # Go through all files directly inside the image folder.
    for file_name in os.listdir(image_folder):

        # Ignore files that are not images.
        if not file_name.lower().endswith(image_extensions):
            continue

        # Remove the image extension.
        file_name_without_extension = os.path.splitext(file_name)[0]

        # Find the object number section in the filename.
        object_position = file_name_without_extension.find("_object_")

        if object_position == -1:
            continue

        # Find the underscore after the object number.
        class_start_position = file_name_without_extension.find(
            "_",
            object_position + len("_object_")
        )

        if class_start_position == -1:
            continue

        # Extract the class name from the filename.
        class_name = file_name_without_extension[
            class_start_position + 1:
        ]

        # Keep the image only if its class belongs to the selected classes.
        if class_name in selected_classes:

            # Store the full image path and its numerical label.
            image_path = os.path.join(image_folder, file_name)
            class_number = class_mapping[class_name]

            manifest_rows.append({
                "image_path": image_path,
                "class_name": class_name,
                "label": class_number
            })

    # Convert the list of records into a DataFrame.
    manifest_df = pd.DataFrame(
        manifest_rows,
        columns=["image_path", "class_name", "label"]
    )

    return manifest_df

# Step 6: Build the training manifest.
training_manifest = create_manifest(
    training_folder,
    selected_class_names,
    class_to_index
)

# Step 7: Build the validation manifest using the same mapping.
validation_manifest = create_manifest(
    validation_folder,
    selected_class_names,
    class_to_index
)

# Step 8: Save both manifests as CSV files.
training_manifest_path = os.path.join(
    training_folder,
    "train_manifest.csv"
)

validation_manifest_path = os.path.join(
    validation_folder,
    "val_manifest.csv"
)

training_manifest.to_csv(
    training_manifest_path,
    index=False,
    encoding="utf-8-sig"
)

validation_manifest.to_csv(
    validation_manifest_path,
    index=False,
    encoding="utf-8-sig"
)

# Step 9: Print the dataset statistics.
print("Number of selected classes:", len(class_to_index))

print("Training images in manifest:", len(training_manifest))
print("Validation images in manifest:", len(validation_manifest))

print("Training images saved to:")
print(training_manifest_path)

print("Validation images saved to:")
print(validation_manifest_path)

# Step 10: Check whether every selected class appears in the training manifest.
training_class_counts = training_manifest["class_name"].value_counts()

classes_missing_from_manifest = [
    class_name
    for class_name in class_to_index
    if class_name not in training_class_counts.index
]

print("Classes missing from training manifest:",
      len(classes_missing_from_manifest))

# Step 11: Check the label range.
if len(training_manifest) > 0:
    print("Smallest training label:", training_manifest["label"].min())
    print("Largest training label:", training_manifest["label"].max())

if len(validation_manifest) > 0:
    print("Smallest validation label:", validation_manifest["label"].min())
    print("Largest validation label:", validation_manifest["label"].max())

In [ ]:
# This code installs the libraries required for dataset analysis and plotting.
get_ipython().run_line_magic('pip', 'install pandas matplotlib pillow')

In [ ]:
# This code creates four dataset quality charts and a text summary.
# It reads the existing CSV files created during dataset preparation.
# It does not change, move, or delete any original training or validation images.

# Part 1: Import the libraries required for data analysis and visualization.
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Part 2: Define the dataset folders and the report output folder.
base_folder = r"."

training_folder = os.path.join(base_folder, "train")
validation_folder = os.path.join(base_folder, "val")

report_folder = os.path.join(
    base_folder,
    "Dataset_Quality_Report"
)

# Part 3: Create the report folder if it does not exist.
os.makedirs(report_folder, exist_ok=True)

# Part 4: Define the paths of the existing CSV files.
distribution_path = os.path.join(
    training_folder,
    "class_distribution.csv"
)

selected_classes_path = os.path.join(
    training_folder,
    "selected_classes.csv"
)

training_manifest_path = os.path.join(
    training_folder,
    "train_manifest.csv"
)

validation_manifest_path = os.path.join(
    validation_folder,
    "val_manifest.csv"
)

# Part 5: Check whether all required CSV files exist.
required_files = [
    distribution_path,
    selected_classes_path,
    training_manifest_path,
    validation_manifest_path
]

for required_file in required_files:
    if not os.path.isfile(required_file):
        raise FileNotFoundError(
            "Required file was not found: " + required_file
        )

print("All required dataset files were found.")

# Part 6: Load the existing dataset statistics and image manifests.
distribution_df = pd.read_csv(distribution_path)
selected_classes_df = pd.read_csv(selected_classes_path)
training_manifest_df = pd.read_csv(training_manifest_path)
validation_manifest_df = pd.read_csv(validation_manifest_path)

# Part 7: Calculate the main dataset statistics.
total_original_training_crops = int(
    distribution_df["image_count"].sum()
)

other_sign_count = int(
    distribution_df.loc[
        distribution_df["class_name"] == "other-sign",
        "image_count"
    ].sum()
)

specific_sign_count = (
    total_original_training_crops - other_sign_count
)

selected_training_count = len(training_manifest_df)
selected_validation_count = len(validation_manifest_df)
selected_class_count = len(selected_classes_df)

# Part 8: Calculate the distribution of the selected training classes.
class_counts = selected_classes_df["image_count"].astype(int)

largest_class_count = int(class_counts.max())
smallest_class_count = int(class_counts.min())
average_class_count = float(class_counts.mean())
median_class_count = float(class_counts.median())

largest_class_name = selected_classes_df.loc[
    selected_classes_df["image_count"].idxmax(),
    "class_name"
]

smallest_class_name = selected_classes_df.loc[
    selected_classes_df["image_count"].idxmin(),
    "class_name"
]

# Part 9: Print a summary of the dataset before generating the charts.
print("\nDATASET SUMMARY")
print("==============================")
print("Original cropped training objects:", total_original_training_crops)
print("other-sign objects:", other_sign_count)
print("Specific sign objects:", specific_sign_count)
print("Selected training images:", selected_training_count)
print("Selected validation images:", selected_validation_count)
print("Selected classes:", selected_class_count)
print("Largest selected class:", largest_class_name, largest_class_count)
print("Smallest selected class:", smallest_class_name, smallest_class_count)
print("Average images per class:", round(average_class_count, 2))
print("Median images per class:", round(median_class_count, 2))

# Part 10: Create Chart 1 to compare the number of images at each stage.
chart1_path = os.path.join(
    report_folder,
    "01_Dataset_Count_Comparison.png"
)

chart1_labels = [
    "All training\ncrops",
    "Specific signs\nexcluding other-sign",
    "Selected\ntraining set",
    "Selected\nvalidation set"
]

chart1_values = [
    total_original_training_crops,
    specific_sign_count,
    selected_training_count,
    selected_validation_count
]

plt.figure(figsize=(10, 6))

bars = plt.bar(
    chart1_labels,
    chart1_values
)

plt.title("Traffic Sign Dataset: Sample Counts", fontsize=14)
plt.ylabel("Number of cropped images", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.3)

for bar, value in zip(bars, chart1_values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.savefig(chart1_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Part 11: Create Chart 2 to show the 15 largest selected classes.
chart2_path = os.path.join(
    report_folder,
    "02_Top_15_Classes.png"
)

top_classes_df = selected_classes_df.sort_values(
    by="image_count",
    ascending=False
).head(15)

top_classes_df = top_classes_df.sort_values(
    by="image_count",
    ascending=True
)

plt.figure(figsize=(12, 8))

plt.barh(
    top_classes_df["class_name"],
    top_classes_df["image_count"]
)

plt.title("Top 15 Traffic Sign Classes in the Training Set", fontsize=14)
plt.xlabel("Number of training images", fontsize=11)
plt.ylabel("Traffic sign class", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(chart2_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Part 12: Create Chart 3 to show the distribution of class sample counts.
chart3_path = os.path.join(
    report_folder,
    "03_Class_Count_Histogram.png"
)

plt.figure(figsize=(10, 6))

plt.hist(
    class_counts,
    bins=25
)

plt.title(
    "Distribution of Training Samples Across Selected Classes",
    fontsize=14
)

plt.xlabel("Number of training images per class", fontsize=11)
plt.ylabel("Number of classes", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(chart3_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Part 13: Create Chart 4 to compare the final training and validation sets.
chart4_path = os.path.join(
    report_folder,
    "04_Training_Validation_Comparison.png"
)

split_labels = [
    "Training set",
    "Validation set"
]

split_values = [
    selected_training_count,
    selected_validation_count
]

plt.figure(figsize=(8, 6))

bars = plt.bar(
    split_labels,
    split_values
)

plt.title("Final Training and Validation Dataset Sizes", fontsize=14)
plt.ylabel("Number of cropped images", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.3)

for bar, value in zip(bars, split_values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.savefig(chart4_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Part 14: Check the readability of a sample of training images.
# This is a small sample check, not a full inspection of all images.
sample_size = min(300, len(training_manifest_df))

sample_paths = training_manifest_df[
    "image_path"
].head(sample_size).tolist()

readable_image_count = 0
unreadable_image_count = 0

sample_widths = []
sample_heights = []

for image_path in sample_paths:
    try:
        with Image.open(image_path) as image:
            image.verify()

        with Image.open(image_path) as image:
            width, height = image.size

        readable_image_count += 1
        sample_widths.append(width)
        sample_heights.append(height)

    except Exception:
        unreadable_image_count += 1

# Part 15: Calculate the validation class coverage.
validation_class_names = set(
    validation_manifest_df["class_name"].dropna().tolist()
)

training_class_names = set(
    training_manifest_df["class_name"].dropna().tolist()
)

classes_missing_from_validation = (
    training_class_names - validation_class_names
)

validation_class_coverage = (
    len(training_class_names) - len(classes_missing_from_validation)
)

# Part 16: Write a text summary that can be used when preparing the report.
summary_path = os.path.join(
    report_folder,
    "Dataset_Quality_Summary.txt"
)

summary_lines = [
    "TRAFFIC SIGN DATASET QUALITY SUMMARY",
    "====================================",
    "",
    "Dataset preparation:",
    "Original training scene images: 36,589",
    "Original validation scene images: 5,320",
    f"All cropped training objects: {total_original_training_crops:,}",
    f"other-sign objects: {other_sign_count:,}",
    f"Specific sign objects excluding other-sign: {specific_sign_count:,}",
    f"Selected classes: {selected_class_count}",
    f"Selected training images: {selected_training_count:,}",
    f"Selected validation images: {selected_validation_count:,}",
    "",
    "Selected training class distribution:",
    f"Average images per class: {average_class_count:.2f}",
    f"Median images per class: {median_class_count:.2f}",
    f"Largest class: {largest_class_name} ({largest_class_count:,})",
    f"Smallest class: {smallest_class_name} ({smallest_class_count:,})",
    "",
    "Validation class coverage:",
    f"Classes represented in validation: {validation_class_coverage}/{len(training_class_names)}",
    f"Classes without validation samples: {len(classes_missing_from_validation)}",
    "",
    "Sample image readability check:",
    f"Images checked: {sample_size}",
    f"Readable images: {readable_image_count}",
    f"Unreadable images: {unreadable_image_count}",
    (
        "Sample width range: "
        + (
            f"{min(sample_widths)} to {max(sample_widths)} pixels"
            if sample_widths else "Not available"
        )
    ),
    (
        "Sample height range: "
        + (
            f"{min(sample_heights)} to {max(sample_heights)} pixels"
            if sample_heights else "Not available"
        )
    ),
    "",
    "Notes:",
    "- The image readability check uses a sample of up to 300 training images.",
    "- The original image files were not changed by this analysis.",
    "- The selected classes exclude other-sign and classes with fewer than 30 training samples.",
    "- The class threshold was determined using training data only.",
]

with open(summary_path, "w", encoding="utf-8") as file:
    file.write("\n".join(summary_lines))

# Part 17: Save the selected class statistics as an additional CSV file.
selected_statistics_path = os.path.join(
    report_folder,
    "Selected_Class_Statistics.csv"
)

selected_classes_df.sort_values(
    by="image_count",
    ascending=False
).to_csv(
    selected_statistics_path,
    index=False,
    encoding="utf-8-sig"
)

# Part 18: Print the locations of all generated report files.
print("\nDATASET QUALITY REPORT COMPLETED")
print("================================")
print("Report folder:", report_folder)
print("Chart 1:", chart1_path)
print("Chart 2:", chart2_path)
print("Chart 3:", chart3_path)
print("Chart 4:", chart4_path)
print("Text summary:", summary_path)
print("Selected class statistics:", selected_statistics_path)

print("\nSample image readability:")
print("Readable:", readable_image_count)
print("Unreadable:", unreadable_image_count)

print("\nValidation classes missing from validation set:",
      len(classes_missing_from_validation))

In [ ]:
# This code checks the deep learning environment and the prepared dataset files.

import os
import pandas as pd
import numpy as np

# Step 1: Define the main folder paths for the prepared dataset.
dataset_folder = r"."

train_manifest_path = os.path.join(
    dataset_folder,
    "train",
    "train_manifest.csv"
)

val_manifest_path = os.path.join(
    dataset_folder,
    "val",
    "val_manifest.csv"
)

class_mapping_path = os.path.join(
    dataset_folder,
    "train",
    "final_class_to_index.json"
)

# Step 2: Check whether the required files exist.
print("Checking dataset files...")
print("----------------------------------------")

print("Training manifest exists:", os.path.exists(train_manifest_path))
print("Validation manifest exists:", os.path.exists(val_manifest_path))
print("Class mapping exists:", os.path.exists(class_mapping_path))

# Step 3: Load the training and validation manifests.
train_df = pd.read_csv(train_manifest_path)
val_df = pd.read_csv(val_manifest_path)

# Step 4: Display the dataset sizes and column names.
print("\nTraining dataset shape:", train_df.shape)
print("Validation dataset shape:", val_df.shape)

print("\nTraining manifest columns:")
print(train_df.columns.tolist())

print("\nValidation manifest columns:")
print(val_df.columns.tolist())

# Step 5: Display a few rows to inspect the file paths and labels.
print("\nFirst five training samples:")
display(train_df.head())

print("\nFirst five validation samples:")
display(val_df.head())

# Step 6: Check the number of unique classes in each dataset.
print("\nNumber of training classes:", train_df["class_name"].nunique())
print("Number of validation classes:", val_df["class_name"].nunique())

print("\nNumber of unique training labels:", train_df["label"].nunique())
print("Number of unique validation labels:", val_df["label"].nunique())

# Step 7: Check whether any image paths are missing.
print("\nChecking image paths...")

train_missing_count = sum(
    not os.path.isfile(image_path)
    for image_path in train_df["image_path"]
)

val_missing_count = sum(
    not os.path.isfile(image_path)
    for image_path in val_df["image_path"]
)

print("Missing training images:", train_missing_count)
print("Missing validation images:", val_missing_count)

# Step 8: Check whether labels contain missing values.
print("\nChecking missing values...")

print("Missing training labels:", train_df["label"].isna().sum())
print("Missing validation labels:", val_df["label"].isna().sum())

# Step 9: Compare the class names in the training and validation sets.
train_classes = set(train_df["class_name"].unique())
val_classes = set(val_df["class_name"].unique())

print("\nClasses found only in training:", len(train_classes - val_classes))
print("Classes found only in validation:", len(val_classes - train_classes))

print("\nDataset inspection completed.")

In [ ]:
# This code checks whether PyTorch is installed and whether an NVIDIA GPU is available.

import sys
import platform

print("Python version:")
print(sys.version)

print("\nOperating system:")
print(platform.platform())

# This section checks whether PyTorch can be imported.
print("\nChecking PyTorch...")

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("PyTorch imported successfully.")

    # This section checks whether CUDA is available for GPU training.
    print("\nChecking CUDA availability...")

    cuda_available = torch.cuda.is_available()

    print("CUDA available:", cuda_available)

    if cuda_available:
        print("Number of available GPUs:", torch.cuda.device_count())

        # This section displays the name of the first available GPU.
        gpu_name = torch.cuda.get_device_name(0)

        print("GPU name:", gpu_name)

        # This section displays the CUDA version used by this PyTorch build.
        print("PyTorch CUDA version:", torch.version.cuda)

        # This section checks the amount of GPU memory.
        gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
        gpu_memory_gb = gpu_memory_bytes / (1024 ** 3)

        print("GPU memory:", round(gpu_memory_gb, 2), "GB")

    else:
        print("No CUDA-enabled GPU is currently available to PyTorch.")
        print("The model can still run on the CPU, but training may be slower.")

except ImportError:
    print("PyTorch is not installed in the current Jupyter environment.")
    print("We need to install a suitable PyTorch version before training.")

In [ ]:
# This code checks the NVIDIA graphics card and the Python environment used by Jupyter.

import sys
import subprocess

# Step 1: Display the Python executable used by the current notebook.
print("Current Python executable:")
print(sys.executable)

# Step 2: Ask the NVIDIA driver to display the graphics card information.
print("\nChecking NVIDIA GPU information...")

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        timeout=15
    )

    if result.returncode == 0:
        print(result.stdout)
    else:
        print("The NVIDIA information command could not run successfully.")
        print(result.stderr)

except FileNotFoundError:
    print("The nvidia-smi command was not found.")
    print("The NVIDIA driver may need to be checked.")

except subprocess.TimeoutExpired:
    print("The NVIDIA information command timed out.")

In [ ]:
# This code verifies that PyTorch is installed and can use the NVIDIA GPU.

import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

    # This section performs a small calculation on the GPU.
    test_tensor = torch.rand(1000, 1000, device="cuda")
    test_result = torch.matmul(test_tensor, test_tensor)

    print("GPU calculation successful.")
    print("Result device:", test_result.device)
else:
    print("GPU is not available to PyTorch yet.")

In [ ]:
# This code defines the basic settings for the traffic sign classification project.

import os
import random
import numpy as np
import torch

# Step 1: Set the random seed to make the experiments more reproducible.
random_seed = 42

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

# Step 2: Define the dataset folder paths.
dataset_folder = r"."

train_manifest_path = os.path.join(
    dataset_folder,
    "train",
    "train_manifest.csv"
)

val_manifest_path = os.path.join(
    dataset_folder,
    "val",
    "val_manifest.csv"
)

# Step 3: Select the device used for model training.
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Step 4: Define the initial model training parameters.
number_of_classes = 326
image_size = 64
batch_size = 64
number_of_epochs = 20
learning_rate = 0.001

# Step 5: Display the settings.
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Number of classes:", number_of_classes)
print("Image size:", image_size, "x", image_size)
print("Batch size:", batch_size)
print("Number of epochs:", number_of_epochs)
print("Learning rate:", learning_rate)
print("Random seed:", random_seed)

In [ ]:
# This code loads the cropped traffic sign images and checks one batch of data.

import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Step 1: Read the training and validation manifests.
train_df = pd.read_csv(train_manifest_path)
val_df = pd.read_csv(val_manifest_path)

# Step 2: Define the image transformation for the baseline model.
# All images are resized to the same size and converted into tensors.
baseline_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

# Step 3: Define a custom dataset class for traffic sign images.
class TrafficSignDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        # Read the image path and its class label from the manifest.
        image_path = self.dataframe.loc[index, "image_path"]
        label = int(self.dataframe.loc[index, "label"])

        # Open the image and convert it to RGB.
        image = Image.open(image_path).convert("RGB")

        # Apply resizing and tensor conversion.
        if self.transform is not None:
            image = self.transform(image)

        return image, label

# Step 4: Create the training and validation datasets.
train_dataset = TrafficSignDataset(
    dataframe=train_df,
    transform=baseline_transform
)

val_dataset = TrafficSignDataset(
    dataframe=val_df,
    transform=baseline_transform
)

# Step 5: Create data loaders for batch processing.
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# Step 6: Load one batch to verify the data.
images, labels = next(iter(train_loader))

print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

print("Minimum label in this batch:", labels.min().item())
print("Maximum label in this batch:", labels.max().item())

print("\nFirst image shape:", images[0].shape)
print("First image label:", labels[0].item())

print("\nData loading test completed successfully.")

In [ ]:
# This code defines the baseline convolutional neural network for traffic sign classification.

import torch
import torch.nn as nn
import torch.optim as optim

# Step 1: Define the CNN model.
class BaselineCNN(nn.Module):

    def __init__(self, number_of_classes=326):
        super(BaselineCNN, self).__init__()

        # This section extracts low-level visual features from input images.
        self.convolutional_layers = nn.Sequential(

            # First convolutional block.
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Second convolutional block.
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Third convolutional block.
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # This section converts the extracted features into class predictions.
        self.classification_layers = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                in_features=128 * 8 * 8,
                out_features=256
            ),

            nn.ReLU(),

            nn.Dropout(p=0.3),

            nn.Linear(
                in_features=256,
                out_features=number_of_classes
            )
        )

    def forward(self, images):

        # Pass the images through the convolutional layers.
        features = self.convolutional_layers(images)

        # Pass the extracted features through the classification layers.
        predictions = self.classification_layers(features)

        return predictions


# Step 2: Create the baseline model and move it to the selected device.
model = BaselineCNN(number_of_classes=number_of_classes)
model = model.to(device)

# Step 3: Define the loss function for multi-class classification.
loss_function = nn.CrossEntropyLoss()

# Step 4: Define the optimizer used to update the model parameters.
optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# Step 5: Count the trainable parameters in the model.
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print("\nDevice:", device)
print("Number of classes:", number_of_classes)
print("Trainable parameters:", f"{trainable_parameters:,}")
print("\nBaseline CNN created successfully.")

In [ ]:
# This code checks whether the baseline CNN produces predictions with the correct shape.

# Step 1: Switch the model to evaluation mode for this test.
model.eval()

# Step 2: Get one batch of images and move it to the selected device.
test_images, test_labels = next(iter(train_loader))

test_images = test_images.to(device)
test_labels = test_labels.to(device)

# Step 3: Disable gradient calculation because this is only a test.
with torch.no_grad():

    test_predictions = model(test_images)

# Step 4: Display the shapes of the input images and model outputs.
print("Input image batch shape:", test_images.shape)
print("Prediction batch shape:", test_predictions.shape)
print("Label batch shape:", test_labels.shape)

# Step 5: Calculate the predicted class for each image.
predicted_classes = torch.argmax(test_predictions, dim=1)

print("\nFirst five predicted labels:")
print(predicted_classes[:5].cpu().tolist())

print("\nFirst five true labels:")
print(test_labels[:5].cpu().tolist())

print("\nForward propagation test completed successfully.")

In [ ]:
# This code trains the baseline CNN and evaluates it on the validation dataset.

import os
import time
import copy
import pandas as pd
import torch

# Step 1: Create a folder for saving model checkpoints and training results.
results_folder = os.path.join(
    dataset_folder,
    "Baseline_CNN_Results"
)

os.makedirs(results_folder, exist_ok=True)

best_model_path = os.path.join(
    results_folder,
    "baseline_cnn_best.pth"
)

training_history_path = os.path.join(
    results_folder,
    "baseline_cnn_training_history.csv"
)

# Step 2: Set the number of training epochs.
number_of_epochs = 20

# Step 3: Create lists for storing the training and validation results.
training_history = []

best_validation_accuracy = 0.0
best_epoch = 0

# Step 4: Record the starting time of the training process.
training_start_time = time.time()

# Step 5: Start the training loop.
for epoch in range(number_of_epochs):

    # This section puts the model into training mode.
    model.train()

    total_training_loss = 0.0
    total_training_correct = 0
    total_training_samples = 0

    # This section processes all batches in the training dataset.
    for images, labels in train_loader:

        # Move the images and labels to the selected device.
        images = images.to(device)
        labels = labels.to(device)

        # Clear the gradients from the previous batch.
        optimizer.zero_grad()

        # Calculate the model predictions.
        predictions = model(images)

        # Calculate the classification loss.
        loss = loss_function(predictions, labels)

        # Calculate gradients using backpropagation.
        loss.backward()

        # Update the model parameters.
        optimizer.step()

        # Accumulate the loss and the number of correct predictions.
        current_batch_size = labels.size(0)

        total_training_loss += loss.item() * current_batch_size

        predicted_labels = torch.argmax(predictions, dim=1)

        total_training_correct += (
            predicted_labels == labels
        ).sum().item()

        total_training_samples += current_batch_size

    # Step 6: Calculate the average training loss and accuracy.
    average_training_loss = (
        total_training_loss / total_training_samples
    )

    training_accuracy = (
        total_training_correct / total_training_samples
    )

    # Step 7: Evaluate the model on the validation dataset.
    model.eval()

    total_validation_loss = 0.0
    total_validation_correct = 0
    total_validation_samples = 0

    # No gradients are required during validation.
    with torch.no_grad():

        for images, labels in val_loader:

            # Move the validation batch to the selected device.
            images = images.to(device)
            labels = labels.to(device)

            # Generate predictions for the validation images.
            predictions = model(images)

            # Calculate validation loss.
            loss = loss_function(predictions, labels)

            current_batch_size = labels.size(0)

            total_validation_loss += (
                loss.item() * current_batch_size
            )

            predicted_labels = torch.argmax(predictions, dim=1)

            total_validation_correct += (
                predicted_labels == labels
            ).sum().item()

            total_validation_samples += current_batch_size

    # Step 8: Calculate the average validation loss and accuracy.
    average_validation_loss = (
        total_validation_loss / total_validation_samples
    )

    validation_accuracy = (
        total_validation_correct / total_validation_samples
    )

    # Step 9: Store the results for this epoch.
    epoch_result = {
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "training_accuracy": training_accuracy,
        "validation_loss": average_validation_loss,
        "validation_accuracy": validation_accuracy
    }

    training_history.append(epoch_result)

    # Step 10: Save the model whenever validation accuracy improves.
    if validation_accuracy > best_validation_accuracy:

        best_validation_accuracy = validation_accuracy
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            best_model_path
        )

    # Step 11: Display the results after this epoch.
    print(
        f"Epoch [{epoch + 1}/{number_of_epochs}] | "
        f"Training Loss: {average_training_loss:.4f} | "
        f"Training Accuracy: {training_accuracy * 100:.2f}% | "
        f"Validation Loss: {average_validation_loss:.4f} | "
        f"Validation Accuracy: {validation_accuracy * 100:.2f}%"
    )

    # Step 12: Save the training history after each epoch.
    history_dataframe = pd.DataFrame(training_history)

    history_dataframe.to_csv(
        training_history_path,
        index=False
    )

# Step 13: Calculate the total training time.
training_end_time = time.time()

total_training_time = (
    training_end_time - training_start_time
)

# Step 14: Display the final training summary.
print("\nBaseline CNN training completed.")

print("Best epoch:", best_epoch)

print(
    "Best validation accuracy:",
    f"{best_validation_accuracy * 100:.2f}%"
)

print(
    "Total training time:",
    f"{total_training_time / 60:.2f} minutes"
)

print("\nBest model saved to:")
print(best_model_path)

print("\nTraining history saved to:")
print(training_history_path)

In [ ]:
# This code checks GPU usage and verifies whether the training process has released resources.

import torch
import subprocess

# Step 1: Check whether CUDA is available.
print("CUDA available:", torch.cuda.is_available())

# Step 2: Display GPU memory currently allocated by PyTorch.
if torch.cuda.is_available():
    allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved_memory = torch.cuda.memory_reserved(0) / (1024 ** 3)

    print("PyTorch allocated GPU memory:", round(allocated_memory, 2), "GB")
    print("PyTorch reserved GPU memory:", round(reserved_memory, 2), "GB")

# Step 3: Display the NVIDIA GPU status.
print("\nNVIDIA GPU status:")

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        timeout=15
    )

    if result.returncode == 0:
        print(result.stdout)
    else:
        print(result.stderr)

except Exception as error:
    print("Could not read GPU status:", error)

In [ ]:
# This code measures the data loading and GPU training speed using 50 batches.

import time
import torch

# Step 1: Make sure the previous training process has stopped.
print("Testing the training speed...")
print("Device:", device)

# Step 2: Set the number of batches for this short test.
number_of_test_batches = 50

# Step 3: Prepare the model for training.
model.train()

# Step 4: Create timing variables.
test_start_time = time.time()
data_loading_time = 0.0
training_step_time = 0.0

# Step 5: Start measuring the data loading and training process.
data_iterator = iter(train_loader)

for batch_number in range(number_of_test_batches):

    # Measure how long it takes to obtain the next batch.
    loading_start_time = time.time()

    try:
        images, labels = next(data_iterator)
    except StopIteration:
        break

    loading_end_time = time.time()

    data_loading_time += loading_end_time - loading_start_time

    # Move the batch to the selected device and synchronize GPU timing.
    training_start_time = time.time()

    images = images.to(device)
    labels = labels.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Perform one normal training step.
    optimizer.zero_grad()

    predictions = model(images)
    loss = loss_function(predictions, labels)

    loss.backward()
    optimizer.step()

    # Wait for GPU operations to finish before measuring elapsed time.
    if device.type == "cuda":
        torch.cuda.synchronize()

    training_end_time = time.time()

    training_step_time += training_end_time - training_start_time

    # Print progress every 10 batches.
    if (batch_number + 1) % 10 == 0:
        elapsed_time = time.time() - test_start_time

        print(
            f"Batch {batch_number + 1}/{number_of_test_batches} | "
            f"Loss: {loss.item():.4f} | "
            f"Elapsed: {elapsed_time:.1f} seconds"
        )

# Step 6: Display the measured speed.
total_test_time = time.time() - test_start_time

print("\nSpeed test completed.")
print("Batches processed:", batch_number + 1)
print("Total test time:", round(total_test_time, 2), "seconds")
print("Total data loading time:", round(data_loading_time, 2), "seconds")
print("Total training step time:", round(training_step_time, 2), "seconds")

if batch_number + 1 > 0:
    average_batch_time = total_test_time / (batch_number + 1)

    print("Average time per batch:", round(average_batch_time, 3), "seconds")

    estimated_epoch_minutes = (
        average_batch_time * len(train_loader) / 60
    )

    print(
        "Estimated training time per epoch:",
        round(estimated_epoch_minutes, 2),
        "minutes"
    )

In [ ]:
# This code resets the baseline CNN so that the formal experiment starts from fresh parameters.

import os
import time
import copy
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# Step 1: Set the random seed for reproducibility.
random_seed = 42

torch.manual_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

# Step 2: Create a fresh baseline CNN model.
model = BaselineCNN(number_of_classes=number_of_classes)

model = model.to(device)

# Step 3: Recreate the loss function.
loss_function = nn.CrossEntropyLoss()

# Step 4: Recreate the optimizer.
# This prevents the previous speed test's optimizer state from being reused.
learning_rate = 0.001

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# Step 5: Create a folder for saving the formal training results.
results_folder = os.path.join(
    dataset_folder,
    "Baseline_CNN_Results"
)

os.makedirs(results_folder, exist_ok=True)

# Step 6: Define the output file paths.
best_model_path = os.path.join(
    results_folder,
    "baseline_cnn_best.pth"
)

training_history_path = os.path.join(
    results_folder,
    "baseline_cnn_training_history.csv"
)

print("Baseline CNN has been reinitialized.")
print("Device:", device)
print("Number of classes:", number_of_classes)
print("Learning rate:", learning_rate)
print("Results folder:", results_folder)

In [ ]:
# This code trains the baseline CNN for one epoch with real-time progress updates.

import time
import pandas as pd
import torch

# Step 1: Set the number of epochs for this initial full training test.
number_of_epochs = 1

# Step 2: Create a list to store the training and validation results.
training_history = []

# Step 3: Initialize the best validation accuracy.
best_validation_accuracy = 0.0
best_epoch = 0

# Step 4: Record the start time.
overall_start_time = time.time()

# Step 5: Switch the model to training mode.
model.train()

total_training_loss = 0.0
total_training_correct = 0
total_training_samples = 0

# Step 6: Calculate the total number of batches.
total_training_batches = len(train_loader)

print("Starting baseline CNN training...")
print("Total training batches:", total_training_batches)
print("Batch size:", batch_size)
print()

# Step 7: Process the training dataset batch by batch.
for batch_number, (images, labels) in enumerate(train_loader):

    # Move the current batch to the GPU or CPU.
    images = images.to(device)
    labels = labels.to(device)

    # Clear the gradients from the previous batch.
    optimizer.zero_grad()

    # Calculate the model predictions.
    predictions = model(images)

    # Calculate the loss.
    loss = loss_function(predictions, labels)

    # Calculate gradients.
    loss.backward()

    # Update the model parameters.
    optimizer.step()

    # Calculate the number of correct predictions.
    predicted_labels = torch.argmax(predictions, dim=1)

    current_batch_size = labels.size(0)

    batch_correct = (
        predicted_labels == labels
    ).sum().item()

    # Accumulate the loss and accuracy statistics.
    total_training_loss += loss.item() * current_batch_size
    total_training_correct += batch_correct
    total_training_samples += current_batch_size

    # Step 8: Print the training progress every 50 batches.
    if (
        (batch_number + 1) % 50 == 0
        or (batch_number + 1) == total_training_batches
    ):

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_time = time.time() - overall_start_time

        current_average_loss = (
            total_training_loss / total_training_samples
        )

        current_accuracy = (
            total_training_correct / total_training_samples
        ) * 100

        progress_percentage = (
            (batch_number + 1) / total_training_batches
        ) * 100

        print(
            f"Progress: {batch_number + 1}/{total_training_batches} "
            f"({progress_percentage:.1f}%) | "
            f"Loss: {current_average_loss:.4f} | "
            f"Accuracy: {current_accuracy:.2f}% | "
            f"Elapsed: {elapsed_time:.1f} seconds"
        )

# Step 9: Calculate the average training loss and accuracy.
average_training_loss = (
    total_training_loss / total_training_samples
)

training_accuracy = (
    total_training_correct / total_training_samples
)

# Step 10: Switch the model to evaluation mode.
model.eval()

total_validation_loss = 0.0
total_validation_correct = 0
total_validation_samples = 0

validation_start_time = time.time()

# Step 11: Evaluate the model on the validation dataset.
with torch.no_grad():

    for batch_number, (images, labels) in enumerate(val_loader):

        # Move the validation batch to the selected device.
        images = images.to(device)
        labels = labels.to(device)

        # Generate predictions.
        predictions = model(images)

        # Calculate the validation loss.
        loss = loss_function(predictions, labels)

        current_batch_size = labels.size(0)

        total_validation_loss += (
            loss.item() * current_batch_size
        )

        predicted_labels = torch.argmax(predictions, dim=1)

        total_validation_correct += (
            predicted_labels == labels
        ).sum().item()

        total_validation_samples += current_batch_size

        # Print validation progress every 50 batches.
        if (
            (batch_number + 1) % 50 == 0
            or (batch_number + 1) == len(val_loader)
        ):
            print(
                f"Validation progress: "
                f"{batch_number + 1}/{len(val_loader)} batches"
            )

# Step 12: Calculate validation metrics.
average_validation_loss = (
    total_validation_loss / total_validation_samples
)

validation_accuracy = (
    total_validation_correct / total_validation_samples
)

# Step 13: Store the results from this epoch.
epoch_result = {
    "epoch": 1,
    "training_loss": average_training_loss,
    "training_accuracy": training_accuracy,
    "validation_loss": average_validation_loss,
    "validation_accuracy": validation_accuracy
}

training_history.append(epoch_result)

# Step 14: Save the model and training history.
best_validation_accuracy = validation_accuracy
best_epoch = 1

torch.save(
    model.state_dict(),
    best_model_path
)

history_dataframe = pd.DataFrame(training_history)

history_dataframe.to_csv(
    training_history_path,
    index=False
)

# Step 15: Display the final results.
overall_end_time = time.time()

print("\nOne-epoch training test completed.")

print(
    "Training loss:",
    f"{average_training_loss:.4f}"
)

print(
    "Training accuracy:",
    f"{training_accuracy * 100:.2f}%"
)

print(
    "Validation loss:",
    f"{average_validation_loss:.4f}"
)

print(
    "Validation accuracy:",
    f"{validation_accuracy * 100:.2f}%"
)

print(
    "Total elapsed time:",
    f"{(overall_end_time - overall_start_time) / 60:.2f} minutes"
)

print("\nModel saved to:")
print(best_model_path)

print("\nTraining history saved to:")
print(training_history_path)

In [ ]:
# This code compares class distributions in the training and validation datasets.

import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Count the number of samples for each class in the training dataset.
train_class_counts = (
    train_df.groupby("class_name")
    .size()
    .reset_index(name="training_count")
)

# Step 2: Count the number of samples for each class in the validation dataset.
val_class_counts = (
    val_df.groupby("class_name")
    .size()
    .reset_index(name="validation_count")
)

# Step 3: Combine the two class-count tables.
class_comparison = pd.merge(
    train_class_counts,
    val_class_counts,
    on="class_name",
    how="outer"
)

# Step 4: Replace missing counts with zero.
class_comparison["training_count"] = (
    class_comparison["training_count"].fillna(0).astype(int)
)

class_comparison["validation_count"] = (
    class_comparison["validation_count"].fillna(0).astype(int)
)

# Step 5: Calculate the validation-to-training sample ratio.
class_comparison["validation_training_ratio"] = (
    class_comparison["validation_count"]
    / class_comparison["training_count"].replace(0, float("nan"))
)

# Step 6: Display summary statistics.
print("Number of classes:", len(class_comparison))

print(
    "Classes with fewer than 30 training samples:",
    (class_comparison["training_count"] < 30).sum()
)

print(
    "Classes with zero validation samples:",
    (class_comparison["validation_count"] == 0).sum()
)

print("\nTraining sample count summary:")
print(class_comparison["training_count"].describe())

print("\nValidation sample count summary:")
print(class_comparison["validation_count"].describe())

# Step 7: Display the classes with the most training samples.
print("\nTop 10 classes by training sample count:")

display(
    class_comparison.sort_values(
        by="training_count",
        ascending=False
    ).head(10)
)

# Step 8: Plot the relationship between training and validation counts.
plt.figure(figsize=(9, 6))

plt.scatter(
    class_comparison["training_count"],
    class_comparison["validation_count"],
    alpha=0.7
)

plt.xlabel("Number of Training Images")
plt.ylabel("Number of Validation Images")
plt.title("Training vs Validation Samples by Class")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# This code trains the baseline CNN from scratch for 20 epochs and saves the results.

import os
import time
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# Step 1: Set the random seed for a reproducible experiment.
random_seed = 42

random.seed(random_seed)
torch.manual_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

# Step 2: Define the number of epochs and the learning rate.
number_of_epochs = 20
learning_rate = 0.001

# Step 3: Reinitialize the model so the formal experiment starts from scratch.
model = BaselineCNN(number_of_classes=number_of_classes)
model = model.to(device)

# Step 4: Define the loss function and optimizer.
loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# Step 5: Create the folder for the formal baseline results.
results_folder = os.path.join(
    dataset_folder,
    "Baseline_CNN_Results"
)

os.makedirs(results_folder, exist_ok=True)

# Step 6: Define separate files for the 20-epoch experiment.
best_model_path = os.path.join(
    results_folder,
    "baseline_cnn_best.pth"
)

final_model_path = os.path.join(
    results_folder,
    "baseline_cnn_final.pth"
)

training_history_path = os.path.join(
    results_folder,
    "baseline_cnn_training_history.csv"
)

# Step 7: Initialize the experiment history and best validation score.
training_history = []
best_validation_accuracy = 0.0
best_epoch = 0

overall_start_time = time.time()

# Step 8: Start training the model for 20 epochs.
for epoch in range(number_of_epochs):

    epoch_start_time = time.time()

    model.train()

    total_training_loss = 0.0
    total_training_correct = 0
    total_training_samples = 0

    total_training_batches = len(train_loader)

    print(
        f"\nStarting Epoch {epoch + 1}/{number_of_epochs}"
    )

    # Step 9: Train on every batch in the training dataset.
    for batch_number, (images, labels) in enumerate(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        predictions = model(images)

        loss = loss_function(predictions, labels)

        loss.backward()

        optimizer.step()

        current_batch_size = labels.size(0)

        predicted_labels = torch.argmax(predictions, dim=1)

        total_training_loss += (
            loss.item() * current_batch_size
        )

        total_training_correct += (
            predicted_labels == labels
        ).sum().item()

        total_training_samples += current_batch_size

        # Step 10: Display progress every 50 batches.
        if (
            (batch_number + 1) % 50 == 0
            or (batch_number + 1) == total_training_batches
        ):

            if device.type == "cuda":
                torch.cuda.synchronize()

            elapsed_time = time.time() - epoch_start_time

            current_average_loss = (
                total_training_loss / total_training_samples
            )

            current_training_accuracy = (
                total_training_correct / total_training_samples
            ) * 100

            progress_percentage = (
                (batch_number + 1) / total_training_batches
            ) * 100

            print(
                f"Epoch {epoch + 1}/{number_of_epochs} | "
                f"Batch {batch_number + 1}/{total_training_batches} "
                f"({progress_percentage:.1f}%) | "
                f"Loss: {current_average_loss:.4f} | "
                f"Accuracy: {current_training_accuracy:.2f}% | "
                f"Time: {elapsed_time:.1f}s"
            )

    # Step 11: Calculate the average training metrics.
    average_training_loss = (
        total_training_loss / total_training_samples
    )

    training_accuracy = (
        total_training_correct / total_training_samples
    )

    # Step 12: Evaluate the model on the validation dataset.
    model.eval()

    total_validation_loss = 0.0
    total_validation_correct = 0
    total_validation_samples = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            predictions = model(images)

            loss = loss_function(predictions, labels)

            current_batch_size = labels.size(0)

            total_validation_loss += (
                loss.item() * current_batch_size
            )

            predicted_labels = torch.argmax(
                predictions,
                dim=1
            )

            total_validation_correct += (
                predicted_labels == labels
            ).sum().item()

            total_validation_samples += current_batch_size

    # Step 13: Calculate the average validation metrics.
    average_validation_loss = (
        total_validation_loss / total_validation_samples
    )

    validation_accuracy = (
        total_validation_correct / total_validation_samples
    )

    # Step 14: Save the metrics for the current epoch.
    epoch_result = {
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "training_accuracy": training_accuracy,
        "validation_loss": average_validation_loss,
        "validation_accuracy": validation_accuracy
    }

    training_history.append(epoch_result)

    # Step 15: Save the best model based on validation accuracy.
    if validation_accuracy > best_validation_accuracy:

        best_validation_accuracy = validation_accuracy
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            best_model_path
        )

    # Step 16: Save the current model and all completed training history.
    torch.save(
        model.state_dict(),
        final_model_path
    )

    history_dataframe = pd.DataFrame(training_history)

    history_dataframe.to_csv(
        training_history_path,
        index=False
    )

    # Step 17: Display the summary for the completed epoch.
    epoch_elapsed_time = time.time() - epoch_start_time

    print(
        f"\nEpoch {epoch + 1} completed | "
        f"Training Loss: {average_training_loss:.4f} | "
        f"Training Accuracy: {training_accuracy * 100:.2f}% | "
        f"Validation Loss: {average_validation_loss:.4f} | "
        f"Validation Accuracy: {validation_accuracy * 100:.2f}% | "
        f"Time: {epoch_elapsed_time / 60:.2f} minutes"
    )

    print(
        "Best validation accuracy so far:",
        f"{best_validation_accuracy * 100:.2f}%"
    )

# Step 18: Display the final experiment summary.
overall_elapsed_time = time.time() - overall_start_time

print("\nBaseline CNN training completed successfully.")

print("Total epochs:", len(training_history))
print("Best epoch:", best_epoch)

print(
    "Best validation accuracy:",
    f"{best_validation_accuracy * 100:.2f}%"
)

print(
    "Total training time:",
    f"{overall_elapsed_time / 60:.2f} minutes"
)

print("\nBest model saved to:")
print(best_model_path)

print("\nFinal model saved to:")
print(final_model_path)

print("\nTraining history saved to:")
print(training_history_path)

In [ ]:
# This code plots the baseline CNN training curves and saves the figures.

import os
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Define the paths to the training history and output folder.
results_folder = os.path.join(
    dataset_folder,
    "Baseline_CNN_Results"
)

training_history_path = os.path.join(
    results_folder,
    "baseline_cnn_training_history.csv"
)

# Step 2: Load the recorded results from all 20 epochs.
history_df = pd.read_csv(training_history_path)

print("Training history loaded successfully.")
print("Number of recorded epochs:", len(history_df))

print("\nLast five epochs:")
display(history_df.tail())

# Step 3: Create the output folder for the training curve figures.
figures_folder = os.path.join(
    results_folder,
    "Training_Curves"
)

os.makedirs(figures_folder, exist_ok=True)

# Step 4: Plot the training loss and validation loss.
plt.figure(figsize=(9, 6))

plt.plot(
    history_df["epoch"],
    history_df["training_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["validation_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline CNN: Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

loss_figure_path = os.path.join(
    figures_folder,
    "01_Baseline_Loss_Curves.png"
)

plt.savefig(
    loss_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Step 5: Plot training accuracy and validation accuracy.
plt.figure(figsize=(9, 6))

plt.plot(
    history_df["epoch"],
    history_df["training_accuracy"] * 100,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    history_df["epoch"],
    history_df["validation_accuracy"] * 100,
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Baseline CNN: Training and Validation Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

accuracy_figure_path = os.path.join(
    figures_folder,
    "02_Baseline_Accuracy_Curves.png"
)

plt.savefig(
    accuracy_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Step 6: Plot validation accuracy and highlight the best epoch.
best_epoch_index = history_df["validation_accuracy"].idxmax()

best_epoch_number = int(
    history_df.loc[best_epoch_index, "epoch"]
)

best_validation_accuracy = (
    history_df.loc[best_epoch_index, "validation_accuracy"] * 100
)

plt.figure(figsize=(9, 6))

plt.plot(
    history_df["epoch"],
    history_df["validation_accuracy"] * 100,
    marker="o",
    label="Validation Accuracy"
)

plt.scatter(
    best_epoch_number,
    best_validation_accuracy,
    s=100,
    label=f"Best Epoch: {best_epoch_number}"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Baseline CNN: Validation Accuracy by Epoch")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

best_epoch_figure_path = os.path.join(
    figures_folder,
    "03_Baseline_Validation_Accuracy.png"
)

plt.savefig(
    best_epoch_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Step 7: Print the saved figure paths and best validation result.
print("\nTraining curves saved successfully.")

print("Loss curves:", loss_figure_path)
print("Accuracy curves:", accuracy_figure_path)
print("Validation accuracy:", best_epoch_figure_path)

print("\nBest epoch:", best_epoch_number)

print(
    "Best validation accuracy:",
    f"{best_validation_accuracy:.2f}%"
)

In [ ]:
# This code creates a data augmentation pipeline and prepares the augmented training loader.

import os
import pandas as pd
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Step 1: Define the image transformations used for data augmentation.
# These transformations are applied randomly when a training image is loaded.
augmentation_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor()
])

# Step 2: Define the transformation for validation images.
# Validation images are not randomly augmented.
validation_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

# Step 3: Reload the training and validation manifests.
train_df = pd.read_csv(train_manifest_path)
val_df = pd.read_csv(val_manifest_path)

# Step 4: Create the augmented training dataset.
augmented_train_dataset = TrafficSignDataset(
    dataframe=train_df,
    transform=augmentation_transform
)

# Step 5: Create the validation dataset without augmentation.
augmented_val_dataset = TrafficSignDataset(
    dataframe=val_df,
    transform=validation_transform
)

# Step 6: Create the training data loader.
augmented_train_loader = DataLoader(
    augmented_train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# Step 7: Create the validation data loader.
augmented_val_loader = DataLoader(
    augmented_val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# Step 8: Check the size of the datasets and one augmented batch.
images, labels = next(iter(augmented_train_loader))

print("Augmented training samples:", len(augmented_train_dataset))
print("Validation samples:", len(augmented_val_dataset))
print("Number of classes:", number_of_classes)

print("\nAugmented image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("\nData augmentation pipeline created successfully.")

In [ ]:
# This code initializes a fresh CNN model for the data augmentation experiment.

import torch
import torch.nn as nn
import torch.optim as optim

# Step 1: Set the random seed for the second experiment.
random_seed = 42

torch.manual_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

# Step 2: Create a new CNN model with the same architecture as the baseline.
augmentation_model = BaselineCNN(
    number_of_classes=number_of_classes
)

augmentation_model = augmentation_model.to(device)

# Step 3: Define the same loss function.
augmentation_loss_function = nn.CrossEntropyLoss()

# Step 4: Define the same optimizer and learning rate.
augmentation_optimizer = optim.Adam(
    augmentation_model.parameters(),
    lr=0.001
)

# Step 5: Define the output folder for the second experiment.
augmentation_results_folder = os.path.join(
    dataset_folder,
    "CNN_Data_Augmentation_Results"
)

os.makedirs(
    augmentation_results_folder,
    exist_ok=True
)

print("Data augmentation model initialized.")
print("Device:", device)
print("Number of classes:", number_of_classes)
print("Learning rate:", 0.001)
print("Results folder:", augmentation_results_folder)

In [ ]:
# This code trains the CNN with data augmentation for 20 epochs and saves the results.

import os
import time
import pandas as pd
import torch

# Step 1: Define the number of training epochs.
augmentation_number_of_epochs = 20

# Step 2: Define the output paths for this experiment.
augmentation_best_model_path = os.path.join(
    augmentation_results_folder,
    "cnn_augmentation_best.pth"
)

augmentation_final_model_path = os.path.join(
    augmentation_results_folder,
    "cnn_augmentation_final.pth"
)

augmentation_history_path = os.path.join(
    augmentation_results_folder,
    "cnn_augmentation_training_history.csv"
)

# Step 3: Initialize the experiment history.
augmentation_training_history = []

augmentation_best_validation_accuracy = 0.0
augmentation_best_epoch = 0

augmentation_overall_start_time = time.time()

# Step 4: Start the training loop.
for epoch in range(augmentation_number_of_epochs):

    augmentation_epoch_start_time = time.time()

    # Step 5: Put the model into training mode.
    augmentation_model.train()

    total_training_loss = 0.0
    total_training_correct = 0
    total_training_samples = 0

    total_training_batches = len(augmented_train_loader)

    print(
        f"\nStarting Augmentation Epoch "
        f"{epoch + 1}/{augmentation_number_of_epochs}"
    )

    # Step 6: Train the model using augmented training images.
    for batch_number, (images, labels) in enumerate(
        augmented_train_loader
    ):

        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from the previous batch.
        augmentation_optimizer.zero_grad()

        # Calculate predictions using the augmented images.
        predictions = augmentation_model(images)

        # Calculate the classification loss.
        loss = augmentation_loss_function(
            predictions,
            labels
        )

        # Calculate gradients and update model parameters.
        loss.backward()
        augmentation_optimizer.step()

        # Calculate training accuracy statistics.
        current_batch_size = labels.size(0)

        predicted_labels = torch.argmax(
            predictions,
            dim=1
        )

        total_training_loss += (
            loss.item() * current_batch_size
        )

        total_training_correct += (
            predicted_labels == labels
        ).sum().item()

        total_training_samples += current_batch_size

        # Step 7: Display progress every 50 batches.
        if (
            (batch_number + 1) % 50 == 0
            or (batch_number + 1) == total_training_batches
        ):

            if device.type == "cuda":
                torch.cuda.synchronize()

            elapsed_time = (
                time.time() - augmentation_epoch_start_time
            )

            current_average_loss = (
                total_training_loss / total_training_samples
            )

            current_training_accuracy = (
                total_training_correct / total_training_samples
            ) * 100

            progress_percentage = (
                (batch_number + 1) / total_training_batches
            ) * 100

            print(
                f"Epoch {epoch + 1}/{augmentation_number_of_epochs} | "
                f"Batch {batch_number + 1}/{total_training_batches} "
                f"({progress_percentage:.1f}%) | "
                f"Loss: {current_average_loss:.4f} | "
                f"Accuracy: {current_training_accuracy:.2f}% | "
                f"Time: {elapsed_time:.1f}s"
            )

    # Step 8: Calculate the average training metrics.
    average_training_loss = (
        total_training_loss / total_training_samples
    )

    training_accuracy = (
        total_training_correct / total_training_samples
    )

    # Step 9: Evaluate the model on the unaugmented validation images.
    augmentation_model.eval()

    total_validation_loss = 0.0
    total_validation_correct = 0
    total_validation_samples = 0

    with torch.no_grad():

        for images, labels in augmented_val_loader:

            images = images.to(device)
            labels = labels.to(device)

            predictions = augmentation_model(images)

            loss = augmentation_loss_function(
                predictions,
                labels
            )

            current_batch_size = labels.size(0)

            total_validation_loss += (
                loss.item() * current_batch_size
            )

            predicted_labels = torch.argmax(
                predictions,
                dim=1
            )

            total_validation_correct += (
                predicted_labels == labels
            ).sum().item()

            total_validation_samples += current_batch_size

    # Step 10: Calculate the average validation metrics.
    average_validation_loss = (
        total_validation_loss / total_validation_samples
    )

    validation_accuracy = (
        total_validation_correct / total_validation_samples
    )

    # Step 11: Record the results for this epoch.
    epoch_result = {
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "training_accuracy": training_accuracy,
        "validation_loss": average_validation_loss,
        "validation_accuracy": validation_accuracy
    }

    augmentation_training_history.append(epoch_result)

    # Step 12: Save the model when validation accuracy improves.
    if (
        validation_accuracy
        > augmentation_best_validation_accuracy
    ):

        augmentation_best_validation_accuracy = (
            validation_accuracy
        )

        augmentation_best_epoch = epoch + 1

        torch.save(
            augmentation_model.state_dict(),
            augmentation_best_model_path
        )

    # Step 13: Save the latest completed model and training history.
    torch.save(
        augmentation_model.state_dict(),
        augmentation_final_model_path
    )

    history_dataframe = pd.DataFrame(
        augmentation_training_history
    )

    history_dataframe.to_csv(
        augmentation_history_path,
        index=False
    )

    # Step 14: Display the completed epoch summary.
    augmentation_epoch_elapsed_time = (
        time.time() - augmentation_epoch_start_time
    )

    print(
        f"\nEpoch {epoch + 1} completed | "
        f"Training Loss: {average_training_loss:.4f} | "
        f"Training Accuracy: {training_accuracy * 100:.2f}% | "
        f"Validation Loss: {average_validation_loss:.4f} | "
        f"Validation Accuracy: {validation_accuracy * 100:.2f}% | "
        f"Time: {augmentation_epoch_elapsed_time / 60:.2f} minutes"
    )

    print(
        "Best validation accuracy so far:",
        f"{augmentation_best_validation_accuracy * 100:.2f}%"
    )

# Step 15: Display the final experiment summary.
augmentation_overall_elapsed_time = (
    time.time() - augmentation_overall_start_time
)

print("\nCNN with data augmentation training completed.")

print("Total epochs:", len(augmentation_training_history))
print("Best epoch:", augmentation_best_epoch)

print(
    "Best validation accuracy:",
    f"{augmentation_best_validation_accuracy * 100:.2f}%"
)

print(
    "Total training time:",
    f"{augmentation_overall_elapsed_time / 60:.2f} minutes"
)

print("\nBest model saved to:")
print(augmentation_best_model_path)

print("\nFinal model saved to:")
print(augmentation_final_model_path)

print("\nTraining history saved to:")
print(augmentation_history_path)

In [ ]:
# This code compares the training histories of the baseline and data augmentation models.

import os
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Define the paths to both experiment folders.
baseline_results_folder = os.path.join(
    dataset_folder,
    "Baseline_CNN_Results"
)

augmentation_results_folder = os.path.join(
    dataset_folder,
    "CNN_Data_Augmentation_Results"
)

# Step 2: Load the training history CSV files.
baseline_history_path = os.path.join(
    baseline_results_folder,
    "baseline_cnn_training_history.csv"
)

augmentation_history_path = os.path.join(
    augmentation_results_folder,
    "cnn_augmentation_training_history.csv"
)

baseline_history_df = pd.read_csv(
    baseline_history_path
)

augmentation_history_df = pd.read_csv(
    augmentation_history_path
)

# Step 3: Create a folder for the final comparison figures.
comparison_folder = os.path.join(
    dataset_folder,
    "Final_Model_Comparison"
)

os.makedirs(
    comparison_folder,
    exist_ok=True
)

# Step 4: Plot the validation accuracy curves.
plt.figure(figsize=(10, 6))

plt.plot(
    baseline_history_df["epoch"],
    baseline_history_df["validation_accuracy"] * 100,
    marker="o",
    label="Baseline CNN"
)

plt.plot(
    augmentation_history_df["epoch"],
    augmentation_history_df["validation_accuracy"] * 100,
    marker="o",
    label="CNN + Data Augmentation"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("Validation Accuracy Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

validation_accuracy_path = os.path.join(
    comparison_folder,
    "01_Validation_Accuracy_Comparison.png"
)

plt.savefig(
    validation_accuracy_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Step 5: Plot the validation loss curves.
plt.figure(figsize=(10, 6))

plt.plot(
    baseline_history_df["epoch"],
    baseline_history_df["validation_loss"],
    marker="o",
    label="Baseline CNN"
)

plt.plot(
    augmentation_history_df["epoch"],
    augmentation_history_df["validation_loss"],
    marker="o",
    label="CNN + Data Augmentation"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

validation_loss_path = os.path.join(
    comparison_folder,
    "02_Validation_Loss_Comparison.png"
)

plt.savefig(
    validation_loss_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Step 6: Calculate the best validation accuracy for each model.
baseline_best_accuracy = (
    baseline_history_df["validation_accuracy"].max() * 100
)

augmentation_best_accuracy = (
    augmentation_history_df["validation_accuracy"].max() * 100
)

baseline_best_epoch = int(
    baseline_history_df.loc[
        baseline_history_df["validation_accuracy"].idxmax(),
        "epoch"
    ]
)

augmentation_best_epoch = int(
    augmentation_history_df.loc[
        augmentation_history_df["validation_accuracy"].idxmax(),
        "epoch"
    ]
)

accuracy_improvement = (
    augmentation_best_accuracy - baseline_best_accuracy
)

# Step 7: Display the final comparison summary.
comparison_results = pd.DataFrame({
    "Model": [
        "Baseline CNN",
        "CNN + Data Augmentation"
    ],
    "Best Validation Accuracy (%)": [
        baseline_best_accuracy,
        augmentation_best_accuracy
    ],
    "Best Epoch": [
        baseline_best_epoch,
        augmentation_best_epoch
    ]
})

print("Final Model Comparison:")
display(comparison_results)

print(
    "\nValidation accuracy improvement:",
    f"{accuracy_improvement:.2f} percentage points"
)

# Step 8: Save the comparison table.
comparison_csv_path = os.path.join(
    comparison_folder,
    "Model_Comparison_Results.csv"
)

comparison_results.to_csv(
    comparison_csv_path,
    index=False
)

# Step 9: Display the saved file paths.
print("\nComparison figures and results saved to:")
print(comparison_folder)

print("\nValidation accuracy figure:")
print(validation_accuracy_path)

print("\nValidation loss figure:")
print(validation_loss_path)

print("\nComparison table:")
print(comparison_csv_path)

In [ ]:
# This code backs up the completed experiments and creates a project status report.

import os
import shutil
import zipfile
from datetime import datetime

# Step 1: Define the main project folder on your computer.
dataset_folder = r"."

# Step 2: Define the folder used to store the project backup.
backup_folder = os.path.join(
    dataset_folder,
    "Project_Backup"
)

os.makedirs(
    backup_folder,
    exist_ok=True
)

# Step 3: Define the folders containing the experiment results.
source_folders = [
    os.path.join(dataset_folder, "Dataset_Quality_Report"),
    os.path.join(dataset_folder, "Baseline_CNN_Results"),
    os.path.join(dataset_folder, "CNN_Data_Augmentation_Results"),
    os.path.join(dataset_folder, "Final_Model_Comparison")
]

# Step 4: Copy each existing experiment folder into the backup folder.
copied_file_count = 0
missing_items = []

for source_folder in source_folders:

    if os.path.isdir(source_folder):

        folder_name = os.path.basename(source_folder)

        destination_folder = os.path.join(
            backup_folder,
            folder_name
        )

        shutil.copytree(
            source_folder,
            destination_folder,
            dirs_exist_ok=True
        )

        # Count the files that were copied into the backup.
        for current_root, directory_names, file_names in os.walk(
            destination_folder
        ):
            copied_file_count += len(file_names)

        print("Backed up folder:", folder_name)

    else:
        missing_items.append(source_folder)
        print("Folder not found:", source_folder)

# Step 5: Copy the important dataset manifests and class mapping files.
important_files = [
    os.path.join(dataset_folder, "train", "train_manifest.csv"),
    os.path.join(dataset_folder, "val", "val_manifest.csv"),
    os.path.join(dataset_folder, "train", "final_class_to_index.json"),
    os.path.join(dataset_folder, "train", "selected_classes.csv"),
    os.path.join(dataset_folder, "train", "class_distribution.csv"),
    os.path.join(dataset_folder, "train", "processing_report.json"),
    os.path.join(dataset_folder, "val", "validation_processing_report.json")
]

manifest_backup_folder = os.path.join(
    backup_folder,
    "Dataset_Manifests"
)

os.makedirs(
    manifest_backup_folder,
    exist_ok=True
)

for source_file in important_files:

    if os.path.isfile(source_file):

        destination_file = os.path.join(
            manifest_backup_folder,
            os.path.basename(source_file)
        )

        shutil.copy2(
            source_file,
            destination_file
        )

        copied_file_count += 1
        print("Backed up file:", os.path.basename(source_file))

    else:
        missing_items.append(source_file)
        print("File not found:", source_file)

# Step 6: Create a text file recording the current project status.
status_file_path = os.path.join(
    backup_folder,
    "PROJECT_STATUS.txt"
)

status_text = """
MTSD TRAFFIC SIGN CLASSIFICATION PROJECT
PROJECT SAVE POINT

DATASET
Training samples: 60,162
Validation samples: 8,736
Number of classes: 326
Class labels: 0 to 325

BASELINE CNN
Completed epochs: 20
Best validation accuracy: 90.66%
Best epoch: 19
Final training accuracy: 95.71%
Final validation accuracy: 90.49%
Training time: 21.04 minutes

DATA AUGMENTATION CNN
Completed epochs: 20
Best validation accuracy: 92.91%
Best epoch: 17
Final training accuracy: 90.02%
Final validation accuracy: 92.79%
Final validation loss: 0.3015
Training time: 46.41 minutes

MODEL COMPARISON
Best validation accuracy improvement: 2.26 percentage points

NEXT TASK
Evaluate both best models on the independent test dataset.
Generate test accuracy, classification reports, and confusion matrices.

IMPORTANT
The cropped images remain in their original folders.
This backup contains experiment outputs and important metadata.
Do not delete the original MTSD_Cropped folder.
"""

with open(
    status_file_path,
    "w",
    encoding="utf-8"
) as status_file:

    status_file.write(status_text)

# Step 7: Create a ZIP archive containing the backup files.
zip_path = os.path.join(
    dataset_folder,
    "MTSD_Project_Backup.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for current_root, directory_names, file_names in os.walk(
        backup_folder
    ):

        for file_name in file_names:

            full_path = os.path.join(
                current_root,
                file_name
            )

            relative_path = os.path.relpath(
                full_path,
                dataset_folder
            )

            zip_file.write(
                full_path,
                relative_path
            )

# Step 8: Display the backup results.
print("\nPROJECT BACKUP COMPLETED")

print("Backup folder:")
print(backup_folder)

print("\nZIP archive:")
print(zip_path)

print("\nNumber of files copied or included:", copied_file_count)

print(
    "ZIP archive size:",
    round(os.path.getsize(zip_path) / (1024 * 1024), 2),
    "MB"
)

print("\nProject status file:")
print(status_file_path)

if missing_items:
    print("\nSome expected items were not found.")
    print("Check the paths above before relying on the backup.")
else:
    print("\nAll expected folders and files were found.")